# Comparative Reliability Performance of N-Body Simulation under Hamiltonian Framework Applying Barnes-Hut Trees
---
Within the realms of modern orbital mechanics application of Barnes-Hut trees is a common practice for large N-Body simulations (Barnes & Hut, 1986). The project describes the creation of a C++ and R tool to simulate the problem assessing reliability of the Barnes-Hut trees for cosmological applications via application of symplectic and non-symplectic integration schemes.

## About The Notebook
This notebook is a simulation prototype, that shows fast results and validation of said results using Python.
It will be kept in a dated entry order expanding each individual feature for a later transfer to C++ and R framework.
It will also comply with the project's gantt chart of progress.

Please bear in mind that the function and variable names follow a certain pattern:
- Integrators and their daughter functions: `operation_integrator_order`
- Mathematical equations take their literal mathematical names, where numerator is a numerator, product is a product, etc.

Displayed equations are numbered consecutively through the whole notebook and are cited as Eq. (*n*). Sources are cited in the Author (Year) form and are listed in the References section at the end of the notebook.

# Symplectic Methods
---

## Setting the Environment and Creating Velocity Verlet 4th Order

In [ ]:
# Imports
# Every module used anywhere in the notebook is imported here, so that no later cell needs an import of its own.
import matplotlib.pyplot as plt
import numpy as np
import random as rnd
import time
import json     # reading experiment files (JSON loader)
import copy     # deep copies of experiment descriptions (JSON loader)
from pathlib import Path        # listing experiment files (runtime)

# Randomisation Seed
seed = 256
rnd.seed(seed)

# Constants
# Yoshida composition constants
yoshida_x0 = -(2**(1/3))/(2-(2**(1/3)))
yoshida_x1 = (1/(2-(2**(1/3))))

# Henon crossing time: the time a typical body needs to cross a cluster in Henon units
# (G = 1, M = 1, E = -1/4). It is the time unit of Experiments 2 and 3.
t_cross = 2.0 * np.sqrt(2.0)

In [ ]:
"""
Creating helper functions to help with force evaluation later.
This cell also holds the minimal Body class and the small vector and copy helpers that are shared by
all the experiments, the initial-condition generators and the JSON runtime, so that each of them is
defined exactly once in the notebook.
"""

# A minimal body: mass, displacement, velocity and acceleration.
# The integrators and the force routines only ever access the attributes m, r, v and a, so any object
# that carries these four attributes can be simulated. All vectors are stored as 3-component NumPy
# arrays, so that the same code works for planar (z = 0) and fully three-dimensional problems.
class Body:
    def __init__(self, m, r, v):
        self.m = m      # mass (scalar)
        self.r = np.array(r, dtype=float)       # displacement vector (x, y, z)
        self.v = np.array(v, dtype=float)       # velocity vector (vx, vy, vz)
        self.a = np.zeros(3)        # acceleration vector, filled in by the integrator on its first call

# Independent copy of a list of bodies, so that every run starts from identical conditions.
# Body() wraps r and v in np.array, which copies them.
# bod_obj – a list of stellar bodies represented as objects
def copy_bodies(bod_obj):
    return [Body(obj.m, obj.r, obj.v) for obj in bod_obj]

# Random unit vector, uniformly distributed over the surface of a sphere
def random_direction():
    cos_t = 2.0 * rnd.random() - 1.0        # cos(polar angle) uniform in [-1, 1]
    phi = 2.0 * np.pi * rnd.random()        # azimuth uniform in [0, 2 pi)
    sin_t = np.sqrt(1.0 - cos_t**2)
    return np.array([sin_t * np.cos(phi), sin_t * np.sin(phi), cos_t])

# Move a system to its centre-of-mass frame (in place): subtract the mass-weighted mean position and
# velocity, so that the system does not drift and its total linear momentum is zero (up to rounding)
# bod_obj – a list of stellar bodies represented as objects
# M – total mass of the system. Defaults to the sum of the masses; the generators pass the nominal
#     total mass instead, which differs from the sum only by rounding.
def to_com_frame(bod_obj, M=None):
    if M is None:
        M = sum(obj.m for obj in bod_obj)
    r_cm = sum(obj.m * obj.r for obj in bod_obj) / M
    v_cm = sum(obj.m * obj.v for obj in bod_obj) / M
    for obj in bod_obj:
        obj.r = obj.r - r_cm
        obj.v = obj.v - v_cm

# Find Euclidian distance between two bodies
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
def euclidian_distance(obj_a, obj_b):
    return np.linalg.norm(obj_a.r - obj_b.r)

# Find initial velocity
# obj – individual object with accessible properties
# G – constant of gravitational attraction
# distance – previously found Euclidian distance
def vel_init(obj, G, distance):
    return rnd.random() * np.sqrt((G*obj.m) / distance)  # random variable in the beginning is used to create random orbit eccentricities (no ejection)

# Get a perpendicular vector
def get_perpendicular(v):
    x, y, z = v
    if x != 0 or y != 0:
        perp = np.array([-y, x, 0.0], dtype=float)
    else:
        perp = np.array([0.0, 1.0, 0.0], dtype=float)

    return perp

# Find velocity direction
# Essentially convert scalar velocity into vector velocity
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
# v – scalar velocity
def vel_direction(obj_a, obj_b, v):
    radius_vector = obj_b.r - obj_a.r
    perp_vector = get_perpendicular(radius_vector)
    unit_perp_vector = perp_vector / np.linalg.norm(perp_vector)    # normalise by its own length

    return v * unit_perp_vector


# Compute Gravitational Acceleration of All Bodies acting on i
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
def acc(G, bod_obj, e):
    N = len(bod_obj)    
    accelerations = [np.zeros(3) for _ in range(N)]     # initialise an array to hold acceleration vectors for all bodies

    # Sigma operation
    # Loop over every body i
    for i in range(N):
        a_i = np.zeros(3)   # net acceleration vector on body i

        # Sum gravitational pull from every other body j
        for j in range(N):
            if i == j:
                continue    # idiot-proofing self-interaction

            # Compute displacement from i to j
            r_ij = bod_obj[i].r - bod_obj[j].r

            # Scalar distance squared
            dist_sq = np.sum(r_ij**2)

            # Denominator with softening factor
            denominator = (dist_sq + e**2)**1.5

            # Add the j-body contribution
            a_i += bod_obj[j].m * (r_ij / denominator)

        # Apply -G and store the net acceleration for body i
        accelerations[i] = -G * a_i

    return accelerations

In [ ]:
"""
Conservation Value Calculations

This cell contains all the functions to help with the conservation and will be further embedded into the code.
Here, we will define later-mentioned Hamiltonian and momenta conservations, as well as their relative values.
Every function implements one equation of the conservation section below, which is cited in its comment.
"""

# Calculate the kinetic energy T of the system (Eq. (10))
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
def t_abs(bod_obj):
    t = 0.0     # total kinetic energy
    # Iterate through the list of bodies to extract individual values for the velocity and mass
    for obj in bod_obj:
        p = obj.m * obj.v       # linear momentum vector of the body
        t += np.dot(p, p) / (2 * obj.m)     # |p|^2 is a dot product, so that the result is a scalar

    return t

# Calculate the (softened) gravitational potential energy V of the system (Eq. (11))
# bod_obj – a list of stellar bodies represented as objects.
# G – the constant of gravitational attraction
# e – softening factor (it has to be the same one as in acc, otherwise H is not the
#     energy that the integrator is actually conserving)
def v_abs(bod_obj, G, e):
    N = len(bod_obj)

    v_tot = 0.0     # potential energy total (without the -G factor)

    # Sum over every UNIQUE pair (i < j): a pair holds one potential energy, not two
    for i in range(N):
        for j in range(i + 1, N):
            r_ij = bod_obj[i].r - bod_obj[j].r      # separation vector
            numerator = bod_obj[i].m * bod_obj[j].m
            denominator = np.sqrt(np.dot(r_ij, r_ij) + e**2)    # |r_ij|^2 is a scalar (dot product)

            v_tot += numerator / denominator

    return -G * v_tot

# Calculate the absolute Hamiltonian value H = T + V (Eq. (12))
# t – system's kinetic energy (calculated previously)
# v – system's potential energy (calculated previously)
def h_abs(t, v):
    return t + v

# Convert the Hamiltonian into a relative energy error H_rel (Eq. (15) with X = H)
# h_0 – initial Hamiltonian value at the start of the simulation clock
# h_n – current Hamiltonian value at the given moment in time
def h_rel(h_0, h_n):
    return abs(h_n - h_0) / abs(h_0)

# Calculate the absolute total linear momentum P, a 3-vector (Eq. (13))
# bod_obj – a list of stellar bodies represented as objects.
def p_abs(bod_obj):
    p_total = np.zeros(3)   # initiate a total linear momentum vector

    # Iterate over every single object in the system
    for obj in bod_obj:
        p_total += obj.m * obj.v

    return p_total

# Characteristic linear momentum of the system, sum_i m_i |v_i|
# Used as the reference P_ref of Eq. (15) when the total momentum is (close to) zero.
def p_scale(bod_obj):
    return sum(obj.m * np.linalg.norm(obj.v) for obj in bod_obj)

# Calculate the total angular momentum L, a 3-vector (Eq. (14))
# bod_obj – a list of stellar bodies represented as objects.
def l_abs(bod_obj):
    l_tot = np.zeros(3)     # initiate the summation vector of the total angular momentum

    # Iterate over every body in the system
    for obj in bod_obj:
        l_tot += obj.m * np.cross(obj.r, obj.v)     # cross product, NOT an element-wise product

    return l_tot

# Characteristic angular momentum of the system, sum_i m_i |r_i x v_i|
# Used as the reference L_ref of Eq. (15) when the total angular momentum is (close to) zero.
def l_scale(bod_obj):
    return sum(obj.m * np.linalg.norm(np.cross(obj.r, obj.v)) for obj in bod_obj)

# Relative error of a conserved VECTOR (Eq. (15) with X = P or X = L): the length of the change divided
# by the length of the initial vector, or by the characteristic size if the initial vector is zero.
# x_0 – initial vector at the start of the simulation
# x_n – current vector at the time of the reading
# scale – characteristic size (p_scale or l_scale at t = 0), used instead of |x_0| if |x_0| ~ 0
def vec_rel(x_0, x_n, scale):
    reference = np.linalg.norm(x_0)
    if reference < 1e-12 * scale:       # X_0 = 0: the relative error is undefined, use the scale
        reference = scale

    return np.linalg.norm(x_n - x_0) / reference

# The relative linear and angular momentum errors P_rel and L_rel are the same function of different
# vectors, so both names refer to vec_rel
p_rel = vec_rel
l_rel = vec_rel

# Snapshot of all conserved quantities at one instant: returns (H, P, L)
def conservation_state(bod_obj, G, e):
    return h_abs(t_abs(bod_obj), v_abs(bod_obj, G, e)), p_abs(bod_obj), l_abs(bod_obj)

In [ ]:
"""
Initial Conditions Generators (08/11/26).
The randomised initial conditions of Experiments 2 and 3 were originally defined inside the experiment
cells. They are collected here, together with the virial rescaling that both of them share, so that the
experiments and the JSON runtime use one and the same implementation. Both generators return a list of
Body objects in the centre-of-mass frame (P_0 = 0, Eq. (13)).
"""


# Plummer sphere sampler (Aarseth, Henon & Wielen 1974), used by Experiment 2
# Draws N bodies of mass M / N whose positions follow the Plummer density profile
#   rho(r) = 3 M / (4 pi a^3) * (1 + r^2 / a^2)^(-5/2)
# and whose speeds follow the matching isotropic distribution function.
# N – number of bodies
# G – the constant of gravitational attraction
# M – total mass of the cluster
# a – Plummer scale length
# r_cut – radius cut-off in units of a (removes the very rare, extremely distant bodies)
def init_plummer(N, G, M, a, r_cut=10.0):
    bod_obj = []

    for _ in range(N):
        # Radius: invert the cumulative mass profile M(r) / M = X  ->  r = a / sqrt(X^(-2/3) - 1)
        while True:
            X = rnd.random()
            if X > 0.0:
                r = a / np.sqrt(X**(-2.0 / 3.0) - 1.0)
                if r < r_cut * a:
                    break

        # Speed: v = q * v_esc, with q drawn from g(q) = q^2 (1 - q^2)^(7/2) by rejection.
        # The maximum of g on [0, 1] is about 0.092, so a box of height 0.1 encloses it.
        while True:
            q, y = rnd.random(), 0.1 * rnd.random()
            if y < q**2 * (1.0 - q**2)**3.5:
                break
        v_esc = np.sqrt(2.0 * G * M / np.sqrt(r**2 + a**2))     # local escape speed of the Plummer potential

        bod_obj.append(Body(M / N, r * random_direction(), q * v_esc * random_direction()))

    # Move to the centre-of-mass frame, so that the cluster does not drift and P_0 = 0 (up to rounding)
    to_com_frame(bod_obj, M)

    return bod_obj


# Uniform random sphere with unequal masses, used by Experiment 3
# N – number of bodies
# G – the constant of gravitational attraction
# M – total mass of the system
# R – radius of the sphere that the bodies are scattered in
# e – softening factor (needed for the potential energy of the virial rescaling)
# delta – relative spread of the masses, m_i uniform in [1 - delta, 1 + delta] before normalisation
# q_vir – target virial ratio Q = T / |V|
def init_random_sphere(N, G, M, R, e, delta, q_vir):
    bod_obj = []

    # Masses: draw the raw values first and then normalise them, so that sum_i m_i = M exactly
    raw_m = [1.0 + delta * (2.0 * rnd.random() - 1.0) for _ in range(N)]
    masses = [M * m / sum(raw_m) for m in raw_m]

    for m in masses:
        # Radius: the mass inside radius r of a uniform sphere grows as r^3, so the cumulative
        # distribution is (r / R)^3 = X, and r = R * X^(1/3) with X uniform in [0, 1]
        r = R * rnd.random()**(1.0 / 3.0)

        # Speed: any distribution would do, because the speeds are rescaled below. Only the
        # directions and the RELATIVE speeds of the bodies remain random.
        speed = rnd.random()

        bod_obj.append(Body(m, r * random_direction(), speed * random_direction()))

    # Move to the centre-of-mass frame, so that the system does not drift and P_0 = 0 (up to rounding)
    to_com_frame(bod_obj, M)

    # Rescale the velocities to the chosen virial ratio. The rescaling is applied after the
    # centre-of-mass correction, so P = 0 is preserved (it is multiplied by the same factor).
    virialise(bod_obj, G, e, q_vir)

    return bod_obj


# Rescale all velocities so that the system has the virial ratio Q = T / |V|, with T and V of
# Eq. (10) and Eq. (11). Q = 0.5 is exact virial equilibrium (2T + V = 0); Q < 0.5 makes the system
# collapse. Random sampling only satisfies the target ratio on average, and the rescaling removes the
# sampling noise. T scales with the square of the velocities, so multiplying every v by
# sqrt(Q |V| / T) gives T = Q |V| exactly.
# bod_obj – a list of stellar bodies represented as objects
# G – the constant of gravitational attraction
# e – softening factor (the same as in the forces, see v_abs)
# q_vir – target virial ratio (default: equilibrium)
def virialise(bod_obj, G, e, q_vir=0.5):
    t, v = t_abs(bod_obj), v_abs(bod_obj, G, e)
    factor = np.sqrt(q_vir * abs(v) / t)
    for obj in bod_obj:
        obj.v = obj.v * factor

In [ ]:
"""
Creating a Velocity Verlet 4th Order Script.
It consists of the acceleration, displacement and velocity computations.

The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
def displ_vv_4(obj, t):
    return obj.r + obj.v*t + (0.5*obj.a*(t**2))

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# a_old – old acceleration of the body
def vel_vv_4(obj, t, a_old):
    return obj.v + 0.5*(a_old+obj.a)*t

# Run Velocity Verlet (4th order, Yoshida composition of three 2nd order steps)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_vv_4(bod_obj, G, e, dt, n_steps, acc_func=acc):
    weights = (yoshida_x1, yoshida_x0, yoshida_x1)    # sub-step sizes as fractions of dt (they sum to 1)

    # Initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):
        # Yoshida sub-steps: each one is a full Velocity Verlet step of size w*dt
        for w in weights:
            h = w * dt

            a_old = [obj.a for obj in bod_obj]    # accelerations before the move

            # Move ALL bodies first (new displacement)
            for obj in bod_obj:
                obj.r = displ_vv_4(obj, h)

            # Re-evaluate the forces once for the whole system (new acceleration)
            for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
                obj.a = a

            # Update ALL velocities with the average of old and new acceleration
            for obj, a_prev in zip(bod_obj, a_old):
                obj.v = vel_vv_4(obj, h, a_prev)

## Velocity Verlet 4th Order Explanation
In order to solve the N-body problem numerically, several quantities have to be evaluated: the Euclidean distance between pairs of bodies, an initial velocity vector obtained from a randomised scalar speed, and the gravitational acceleration, which is then propagated in time by the integrator. The initial speed is scaled by a random factor in order to produce a variety of orbital eccentricities for experimental purposes.

**Notation.** Bodies are indexed by $i, j = 1, \dots, N$. Vectors carry a hat in the convention used throughout this notebook: $\hat{r}_i$ is the displacement, $\hat{v}_i$ the velocity and $\hat{a}_i$ the acceleration of body $i$, while $m_i$ is its mass. $G$ is the gravitational constant, $\delta{t}$ is the timestep, $\epsilon$ is the softening length (`e` in the code) and $\xi$ is the random eccentricity factor.

### Euclidean Distance in 3D
The separation of two bodies is the length of the difference of their displacement vectors (`euclidian_distance`, Eq. (1)):

$$r_{ij}=\lvert\hat{r}_i-\hat{r}_j\rvert=\sqrt{(x_i-x_j)^2 + (y_i - y_j)^2 + (z_i-z_j)^2} \tag{1}$$

### Initial Speed
The speed of a circular orbit at the distance $r_{ij}$ of Eq. (1) around a body of mass $M$ is $v_c=\sqrt{GM/r_{ij}}$, where $M$ is the mass of the attracting body. The initial speed is a random fraction of it (`vel_init`, Eq. (2)):

$$v_0=\xi\sqrt{\frac{GM}{r_{ij}}},\qquad \xi\sim\mathcal{U}[0,1) \tag{2}$$

The velocity is launched perpendicular to the radius vector, so the launch point is an apsis of the resulting orbit. For $\xi<1$ the body starts at apocentre and follows an ellipse of eccentricity $e_{\mathrm{orb}}=1-\xi^2$, while $\xi\to1$ gives a circular orbit. Since the escape speed $\sqrt{2}\,v_c$ is never reached, no body is ejected at the start of the simulation.

### Initial Velocity Direction
The scalar speed is converted into a vector by choosing a direction perpendicular to the radius vector $\Delta\hat{r}=\hat{r}_j-\hat{r}_i=(\Delta x,\Delta y,\Delta z)$ (`get_perpendicular` and `vel_direction`, Eq. (3)):

$$\hat{p}=(-\Delta y,\ \Delta x,\ 0),\qquad \hat{n}=\frac{\hat{p}}{\lvert\hat{p}\rvert},\qquad \hat{v}_0=v_0\,\hat{n} \tag{3}$$

The vector $\hat{p}$ is perpendicular to $\Delta\hat{r}$ because $\hat{p}\cdot\Delta\hat{r}=-\Delta x\Delta y+\Delta y\Delta x=0$. If $\Delta x=\Delta y=0$ the construction degenerates, and $\hat{p}=(0,1,0)$ is used instead. As $\hat{n}$ lies in the $xy$-plane, the initial orbits are planar.

### Gravitational Acceleration
Newton's law of gravitation, summed over all other bodies, gives the acceleration of body $i$ (`acc`, Eq. (4)). A softening length $\epsilon$ is added to the denominator (Plummer softening; Plummer, 1911; Dehnen & Read, 2011) so that the force stays finite when two bodies approach each other:

$$\hat{a}_i=-G\sum_{j\neq i}\frac{m_j\,(\hat{r}_i-\hat{r}_j)}{\left(\lvert\hat{r}_i-\hat{r}_j\rvert^2+\epsilon^2\right)^{3/2}} \tag{4}$$

This direct summation requires $\mathcal{O}(N^2)$ pair evaluations per call, which is precisely the cost that the Barnes-Hut tree reduces to $\mathcal{O}(N\log N)$ in the later C++ implementation.

### Velocity Verlet (Second Order)
Since the algorithm is a 4th-order Velocity Verlet, it integrates with a different pattern to the regular Verlet (Verlet, 1967).
Regular 2nd-order Velocity Verlet (Swope et al., 1982) uses the evaluation in a following pattern:
- Set up initial conditions
- Evaluate displacement
- Evaluate acceleration
- Evaluate velocity

I coined a name for this method, RAV, due to $\hat{r}$ being displacement, $\hat{a}$ acceleration and $\hat{v}$ velocity. Velocity Verlet can be expressed via the following equations, where a single step is denoted $S_2(\delta{t})$:

**Displacement** (`displ_vv_4`, Eq. (5))

$$\hat{r}(t+\delta{t}) = \hat{r}(t)+\hat{v}(t)\,\delta{t}+\frac{\hat{a}(t)\,\delta{t}^2}{2} \tag{5}$$

**Velocity** (`vel_vv_4`, Eq. (6))

$$\hat{v}(t+\delta{t})=\hat{v}(t)+\frac{\hat{a}(t)+\hat{a}(t+\delta{t})}{2}\,\delta{t} \tag{6}$$

The new acceleration $\hat{a}(t+\delta{t})$ in Eq. (6) is evaluated from the new displacements of Eq. (5), which is why the order of the three operations cannot be changed. The scheme is time-reversible and symplectic, but only second-order accurate: its global error scales as $\mathcal{O}(\delta{t}^2)$ (Hairer, Lubich & Wanner, 2006).

### Yoshida Composition to 4th Order
Yoshida's method (Yoshida, 1990; see also Forest & Ruth, 1990) raises the order by composing three second-order steps of different sizes (Eq. (7)):

$$S_4(\delta{t})=S_2(w_1\delta{t})\circ S_2(w_0\delta{t})\circ S_2(w_1\delta{t}) \tag{7}$$

$$w_1=\frac{1}{2-2^{1/3}}\approx 1.3512,\qquad w_0=-\frac{2^{1/3}}{2-2^{1/3}}\approx -1.7024 \tag{8}$$

In the code the weights of Eq. (8) are `yoshida_x1` and `yoshida_x0`. They are fixed by the two conditions of Eq. (9): the sub-steps must add up to one full step, and the leading error term of $S_2$ must cancel,

$$2w_1+w_0=1,\qquad 2w_1^3+w_0^3=0 \tag{9}$$

so that the local error falls from $\mathcal{O}(\delta{t}^3)$ to $\mathcal{O}(\delta{t}^5)$ and the global error to $\mathcal{O}(\delta{t}^4)$. The middle weight $w_0$ is negative, so the middle sub-step runs backwards in time, but the three sub-steps together advance the system by exactly $\delta{t}$, and the composition remains symplectic. The price is three force evaluations per timestep instead of one.

### One Integration Step in the Code
For each of the three sub-steps of size $h=w\,\delta{t}$ with $w\in\{w_1, w_0, w_1\}$ (`run_vv_4`):
1. Store the current accelerations $\hat{a}_{\mathrm{old}}$ of all bodies.
2. Move all bodies with $\delta{t}\to h$ (Eq. (5)).
3. Re-evaluate the accelerations of all bodies at the new positions (Eq. (4)).
4. Update all velocities with $\delta{t}\to h$ and $\hat{a}_{\mathrm{old}}$ (Eq. (6)).

All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Conservation Formulae
## Energy (Hamiltonian) Conservation

Energy conservation is a widely used metric to directly track and diagnose issues within the numerical integrators. Due to the nature of symplectic integrators, the energy error does not drift away steadily but oscillates in a bounded band around the true value, which is the signature of the symplectic property (Hairer, Lubich & Wanner, 2006). Non-symplectic methods may show a short-term success, whereas after a larger number of iterations their energy error grows systematically.

### Absolute Energy Conservation
Kinetic energy $T$ is expressed via

$$T=\sum^N_{i=1}\frac{|\hat{p}_i|^2}{2m_i} \tag{10}$$

while potential energy $V$ is a sum over every *unique* pair of bodies:

$$V=-G\sum_{1\leq i < j \leq N} \frac{m_i m_j}{\sqrt{|\hat{r}_i - \hat{r}_j|^2 + \epsilon^2}} \tag{11}$$

where $\hat{p}_i = m_i \hat{v}_i$ and $|\hat{r}_i - \hat{r}_j|$ is the Euclidean distance between two bodies, Eq. (1). The strict inequality $i<j$ matters: a pair holds a single potential energy, so summing over $i\neq j$ (or $i\leq j$) would double-count it (or include self-interaction). The softening length $\epsilon$ is the same as in the acceleration, since $\hat{a}_i=-\frac{1}{m_i}\nabla_{\hat{r}_i}V$ reproduces Eq. (4) exactly.

Hence the Hamiltonian $H$ is the sum of Eq. (10) and Eq. (11):

$$H(\hat{r},\hat{p})=T(\hat{p})+V(\hat{r}) \tag{12}$$

## Momentum Calculus
Momentum calculations for both Linear $\hat{P}$ and Angular $\hat{L}$ momenta are required, since the momenta have to stay constant alongside with the oscillating or near-constant energy. Meaning, deviations from the constant may indicate underlying issues with the integrators or the simulation loops. All of the above-mentioned parameters allow us to track accuracy.

### Total Linear Momentum $\hat{P}$
Total linear momentum is simply the vector sum of individual linear momenta across all bodies and expressed like this:
$$
\hat{P}=\sum^N_{i=1}m_i\hat{v}_i \tag{13}
$$

It is a 3-vector, hence each component from $x$ to $z$ inclusively is summed separately.

### Angular Momentum $\hat{L}$
$$
\hat{L} = \sum^N_{i=1}m_i(\hat{r}_i\times\hat{v}_i) \tag{14}
$$

The angular momentum is the other part of the momenta conservation toolkit. It is a vector built from the **cross product** of displacement and velocity (not the product of their unit vectors, and not a component-wise product). Both linear and angular momenta are conserved by the exact dynamics, and for pairwise central forces a symplectic integrator preserves them to rounding error (Hairer, Lubich & Wanner, 2006), which makes them a sharp test for coding mistakes in the force routine and the loop.

## Relative Errors $H_{rel}$, $P_{rel}$ and $L_{rel}$
The absolute values of Eq. (12), Eq. (13) and Eq. (14) depend on the size of the system, so all three quantities are tracked through one common relative error, which normalises the change since the start of the simulation by a reference value. For a conserved quantity $X\in\{H,\hat{P},\hat{L}\}$ evaluated at step $n$, with the initial value $X_0$,

$$X_{rel}=\frac{\lvert X_n-X_0\rvert}{X_{ref}},\qquad X_{ref}=\lvert X_0\rvert \tag{15}$$

where $\lvert\cdot\rvert$ is the absolute value for the scalar $H$ and the length for the vectors $\hat{P}$ and $\hat{L}$. Eq. (15) yields the relative energy error $H_{rel}$, which is the main tracking tool of the energy conservation, and the relative linear and angular momentum errors $P_{rel}$ and $L_{rel}$.

Systems that are set up in their centre-of-mass frame (such as the figure-eight of Experiment 1; Moore, 1993; Chenciner & Montgomery, 2000) have $\hat{P}_0=0$, and the figure-eight also has $\hat{L}_0=0$, so that Eq. (15) is undefined for them. In that case the reference is replaced by the characteristic size of the individual contributions, $P_{ref}=\sum_i m_i\lvert\hat{v}_i(0)\rvert$ and $L_{ref}=\sum_i m_i\lvert\hat{r}_i(0)\times\hat{v}_i(0)\rvert$, so that the error remains relative to the size of the individual momenta. No such fallback is required for $H$, since $H_0<0$ for every bound system simulated in this notebook.

In [ ]:
"""
Experiment 1: the three-body figure-eight choreography (28/10/26).

Purpose:
    The figure-eight is a periodic solution of the planar three-body problem in which
    three equal masses chase one another along a single closed curve shaped like the
    number eight. It was found numerically by Moore (1993) and its existence was proven
    by Chenciner & Montgomery (2000). Because the exact answer is known (the bodies must
    return to their starting positions after every period T), it is an ideal first
    validation test for the integrator: any drift away from the starting positions after
    a whole number of periods is a direct measure of the numerical error.

Setup:
    - three equal masses chasing each other along a single figure-eight curve
    - initial conditions from Simo's high-precision values, in natural units G = m = 1
    - no softening: the bodies never come close, and softening would perturb the orbit
    - integrated with run_vv_4 (Velocity Verlet 4th order)

Time control:
    - the total simulated time is a whole number of orbital periods
    - dt is dictated by the number of timesteps: dt = t_total / n_steps
    - frames are computed separately: one trajectory point is stored every
      steps_per_frame = n_steps / n_frames steps
Conservation tracking:
    The Hamiltonian H and the linear and angular momenta P and L are evaluated at every stored
    frame (conservation_state), and their relative errors H_rel, P_rel and L_rel are plotted
    together with the orbit in 3D and its 2D projection onto the x-y plane. The output is these five
    plots plus the closure error after the run.
"""


# The bodies are instances of the Body class of the helpers cell (mass, displacement, velocity and
# acceleration as 3-component NumPy arrays).


# Experiment parameters (natural units: G = 1, m = 1)
# The published figure-eight initial conditions are dimensionless: they assume G = 1
# and unit masses. Working in SI units would require rescaling every quantity, so the
# experiment is run directly in these natural units. Lengths, times and velocities
# printed below are therefore dimensionless as well.
G_nat = 1.0     # gravitational constant in natural units

# Plummer softening (Eq. 4) is switched off. The closest approach between any two
# bodies on the figure-eight is about 0.7 length units, so there is no risk of a
# division by zero, and any non-zero softening would alter the force law and hence
# destroy the periodicity that this experiment relies on.
e_soft = 0.0    # softening length (switched off, see above)

# Period of one full figure-eight orbit in natural units. After exactly one period
# every body returns to its own starting position. The value is only known to a
# finite number of digits, which sets a lower floor on the closure error below.
T_period = 6.32591398     # period of the figure-eight orbit (Simo's value, 9 significant digits)

# The three numbers below fully control the time discretisation:
#   n_periods – how long the simulation runs (t_total = n_periods * T_period)
#   n_steps – how finely that time is divided (this alone dictates dt)
#   n_frames – how many snapshots are stored for plotting (independent of dt)
# Increasing n_steps makes dt smaller and the result more accurate (the global error
# of the 4th-order scheme falls as dt^4) at a proportionally higher cost, while
# n_frames only changes the resolution of the plots and the memory used.
n_periods = 3   # number of full orbits to simulate
n_steps = 30000     # total number of timesteps (dictates dt)
n_frames = 1500     # number of stored trajectory points (independent of dt)

# Derived time quantities
t_total = n_periods * T_period      # total simulated time
dt = t_total / n_steps      # timestep implied by the chosen number of steps

# The integration is split into n_frames equal blocks of steps_per_frame steps each,
# and one snapshot is stored at the end of every block. For the blocks to add up to
# exactly n_steps, n_steps must be an integer multiple of n_frames; otherwise the run
# would stop short of (or overshoot) t_total and the closure test would be invalid.
assert n_steps % n_frames == 0, "n_steps must be divisible by n_frames"
steps_per_frame = n_steps // n_frames       # integration steps between two stored snapshots

# Initial conditions
# The configuration at t = 0 is symmetric:
#   – body 2 sits exactly at the crossing point (centre) of the eight
#   – bodies 1 and 3 sit at mirror-image positions, r3 = -r1
#   – bodies 1 and 3 share the same velocity, and body 2 moves with -2 times it
# These conditions give zero total momentum (m*v1 + m*v2 + m*v3 = 0), so the centre of
# mass stays fixed at the origin and no correction for drift is needed (unlike in a
# system with a randomly launched star). All z-components are zero, so the motion
# remains in the x-y plane for the entire run.
r1 = [0.97000436, -0.24308753, 0.0]     # displacement of body 1 (body 3 is its mirror image)
v2 = [-0.93240737, -0.86473146, 0.0]    # velocity of the central body 2
v13 = [-0.5 * v2[0], -0.5 * v2[1], 0.0]     # v1 = v3 = -v2 / 2, so the total momentum is zero

# The three bodies, each with unit mass
bodies = [
    Body(1.0, r1, v13),     # body 1: right-hand lobe
    Body(1.0, [0.0, 0.0, 0.0], v2),     # body 2: centre of the eight
    Body(1.0, [-r1[0], -r1[1], 0.0], v13),      # body 3: left-hand lobe
]

# Copies of the starting positions, kept for the closure test at the end of the run.
# .copy() is essential: without it r_start would refer to the same arrays that the
# integrator overwrites, and the closure error would always read zero.
r_start = [b.r.copy() for b in bodies]

# Summary of the time discretisation actually used
print(f"t_total = {t_total:.4f} ({n_periods} periods), n_steps = {n_steps}, dt = {dt:.3e}")
print(f"n_frames = {n_frames}, steps_per_frame = {steps_per_frame}")

# Conservation references
# The state at t = 0 is the reference for every relative error. P_0 and L_0 are exactly
# zero for this configuration (see the initial-conditions comment above), so p_rel and
# l_rel fall back on the characteristic size of the individual momenta, p_scale and l_scale.
H_0, P_0, L_0 = conservation_state(bodies, G_nat, e_soft)
P_scale, L_scale = p_scale(bodies), l_scale(bodies)
print(f"H_0 = {H_0:.6f}, |P_0| = {np.linalg.norm(P_0):.1e}, |L_0| = {np.linalg.norm(L_0):.1e}")

# Simulation
# trajectories[k] is the list of stored displacements of body k. It starts with the
# initial position, so after the loop each list holds n_frames + 1 points (t = 0 plus
# one point per block). run_vv_4 advances the bodies in place by steps_per_frame steps
# of size dt; after each call the new positions are copied into the record and the
# conserved quantities are measured on the freshly updated bodies.
# NB: run_vv_4 recomputes the accelerations at the start of every call. This costs one
# extra force evaluation per frame but does not change the result, because the
# accelerations depend only on the current positions.
trajectories = [[b.r.copy()] for b in bodies]
h_err, p_err, l_err = [0.0], [0.0], [0.0]       # relative errors at t = 0 are zero by definition
for _ in range(n_frames):
    run_vv_4(bodies, G_nat, e_soft, dt, steps_per_frame)
    for k, b in enumerate(bodies):
        trajectories[k].append(b.r.copy())

    H_n, P_n, L_n = conservation_state(bodies, G_nat, e_soft)
    h_err.append(h_rel(H_0, H_n))
    p_err.append(p_rel(P_0, P_n, P_scale))
    l_err.append(l_rel(L_0, L_n, L_scale))

# Convert the nested lists into a single array for convenient slicing:
#   axis 0 – body index k, axis 1 -> frame index, axis 2 -> coordinate (x, y, z)
trajectories = np.array(trajectories)       # shape: (N bodies, n_frames + 1, 3)
h_err, p_err, l_err = np.array(h_err), np.array(p_err), np.array(l_err)
t_frames = np.arange(n_frames + 1) * steps_per_frame * dt / T_period    # time in orbital periods

# Validation
# Closure check: after a whole number of periods every body should be back where it
# started. The distance |r(t_total) - r(0)| therefore measures the accumulated error.
# It contains two contributions: the integration error of the scheme (which shrinks as
# dt^4 when n_steps is increased) and the finite precision of T_period and of the
# initial conditions (which does not shrink with dt). Once the error stops decreasing
# when n_steps is raised, the second contribution has become dominant.
for k, b in enumerate(bodies):
    print(f"body {k + 1}: closure error |r(T) - r(0)| = {np.linalg.norm(b.r - r_start[k]):.3e}")

# Planarity check: with all z-components initially zero, the accelerations of Eq. (4)
# have no z-component either, so z must remain exactly zero for the whole run.
print(f"max |z| over the run = {np.abs(trajectories[:, :, 2]).max():.3e} (the orbit is planar)")

# Conservation summary
print(f"max H_rel = {h_err.max():.3e}, max P_rel = {p_err.max():.3e}, max L_rel = {l_err.max():.3e}")

# Plots: 3D orbit + 2D orbit + the three relative conservation errors
colours = plt.cm.tab10(np.arange(3))        # one distinct colour per body

# All three bodies travel along the very same curve, so lines of equal width would hide
# one another. Drawing them with decreasing widths (body 1 widest, body 3 thinnest)
# keeps every trajectory visible as a coloured band inside the previous one.
line_widths = [3.0, 1.8, 0.8]
fig = plt.figure(figsize=(16, 11))

# Layout on a 2 x 6 grid: the two orbit views share the top row (3 columns each) and the
# three conservation panels share the bottom row (2 columns each).
grid = fig.add_gridspec(2, 6)

# 3D view
# The orbit is planar, so this view shows the eight lying flat at z = 0. Any spurious
# out-of-plane motion would be visible here. The end markers (o) should lie on top of
# the start markers (x) because the run covers a whole number of periods.
ax3d = fig.add_subplot(grid[0, 0:3], projection="3d")
for k in range(len(bodies)):
    x, y, z = trajectories[k].T     # transpose (frames, 3) three coordinate arrays
    ax3d.plot(x, y, z, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax3d.scatter(x[0], y[0], z[0], color=colours[k], marker="x", s=30)     # starting position
    ax3d.scatter(x[-1], y[-1], z[-1], color=colours[k], s=30)       # final position

# Use the same limits on all three axes and a cubic box, so that lengths in different
# directions are directly comparable and the orbit is not visually distorted.
lim = 1.1 * np.abs(trajectories).max()      # 10% margin around the furthest point reached
ax3d.set_xlim(-lim, lim)
ax3d.set_ylim(-lim, lim)
ax3d.set_zlim(-lim, lim)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title("Figure-eight orbit in 3D (x = start, o = end)")
ax3d.legend(fontsize=8, loc="upper left")

# 2D view
# The motion is planar, so the projection onto the x-y plane shows the true shape of the
# orbit without the perspective distortion of the 3D view. An equal aspect ratio keeps the
# eight undistorted, so that its two lobes can be compared directly.
ax2d = fig.add_subplot(grid[0, 3:6])
for k in range(len(bodies)):
    x, y = trajectories[k, :, 0], trajectories[k, :, 1]
    ax2d.plot(x, y, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax2d.scatter(x[0], y[0], color=colours[k], marker="x", s=40, zorder=3)     # starting position
    ax2d.scatter(x[-1], y[-1], color=colours[k], s=40, zorder=3)       # final position
ax2d.set_xlim(-lim, lim)
ax2d.set_ylim(-lim, lim)
ax2d.set_aspect("equal")
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Figure-eight orbit in the x-y plane (x = start, o = end)")
ax2d.legend(fontsize=8, loc="upper left")
ax2d.grid(alpha=0.3)

# Relative conservation errors (logarithmic y-axis, time measured in orbital periods).
# The floor of 1e-18 only keeps exact zeros (e.g. at t = 0) drawable on a log axis.
floor = 1e-18
panels = [
    (h_err, r"$H_{rel}=|H_n-H_0|/|H_0|$", "Relative Hamiltonian error", "tab:blue"),
    (p_err, r"$P_{rel}=|\hat{P}_n-\hat{P}_0|/\sum_i m_i|\hat{v}_i(0)|$", "Relative linear momentum error", "tab:green"),
    (l_err, r"$L_{rel}=|\hat{L}_n-\hat{L}_0|/\sum_i m_i|\hat{r}_i(0)\times\hat{v}_i(0)|$", "Relative angular momentum error", "tab:red"),
]
for idx, (err, ylabel, title, colour) in enumerate(panels):
    ax = fig.add_subplot(grid[1, 2 * idx:2 * idx + 2])
    ax.semilogy(t_frames, np.maximum(err, floor), color=colour, lw=1.2)
    ax.set_xlabel("time (orbital periods)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, which="both")

fig.suptitle(f"Three-body figure-eight: {n_periods} periods, {n_steps} steps, "
             f"dt = {dt:.2e} (Velocity Verlet 4th order)")
plt.tight_layout()
plt.show()

## Creating the Barnes-Hut Octree
The direct summation of Eq. (4) evaluates every pair of bodies and therefore scales as $\mathcal{O}(N^2)$. The cell below implements the Barnes-Hut octree (Barnes & Hut, 1986), which approximates the same acceleration in $\mathcal{O}(N\log N)$ operations. The routine `acc_bh` is written as a drop-in replacement for `acc`, and `run_vv_4` accepts either of them through its `acc_func` argument, so that the integrator itself remains unchanged.

In [ ]:
"""
Creating the Barnes-Hut Octree (30/10/26).
It consists of the tree construction, the mass moments of every node and the tree walk
that evaluates the approximate gravitational acceleration of every body.

acc_bh has exactly the same signature as the direct summation acc (plus the opening
angle theta), so it can be handed to run_vv_4 through its acc_func argument.
The detailed description of the algorithm will be provided later down below.

The last three functions of the cell are tree utilities shared by Experiments 2 and 3 and by the
JSON runtime: acc_bh_theta (fixes the opening angle), force_error_bh (accuracy against direct
summation) and tree_stats_bh (size and depth of the octree).
"""

# Maximum depth of the tree. Two bodies sitting at (almost) the same position can never be
# separated by subdividing the cube, so without this limit the insertion would recurse
# forever. At depth 32 a cube is 2^32 times smaller than the root, far below any physical
# separation in these simulations, so the limit only ever acts as a safety net.
max_depth_bh = 32


# A single node of the octree: one cubic cell of space.
# centre – geometric centre of the cube (3-vector)
# half – half of the side length of the cube (scalar)
# m – total mass of all bodies inside the cube (filled in by moments_bh)
# com – centre of mass of all bodies inside the cube (filled in by moments_bh)
# bodies – indices of the bodies stored in this node (only used while it is a leaf)
# children – list of 8 child nodes once the cube has been subdivided, None for a leaf
class Node:
    def __init__(self, centre, half):
        self.centre = np.array(centre, dtype=float)
        self.half = half
        self.m = 0.0
        self.com = np.zeros(3)
        self.bodies = []
        self.children = None


# Find the octant (0-7) of a node that contains the displacement r.
# Each bit of the index answers one question: bit 0 – is x on the positive side of the
# centre, bit 1 – is y, bit 2 – is z. Hence octant = [x >= c_x] + 2[y >= c_y] + 4[z >= c_z].
# node – the node being subdivided
# r – displacement vector of the body
def octant_bh(node, r):
    return int(r[0] >= node.centre[0]) + 2 * int(r[1] >= node.centre[1]) + 4 * int(r[2] >= node.centre[2])


# Split a leaf into 8 equal child cubes of half the side length.
# The child with index k is shifted from the parent centre by half/2 in every direction,
# with the sign of each shift taken from the corresponding bit of k (same rule as octant_bh).
# node – the leaf to be subdivided
def subdivide_bh(node):
    quarter = 0.5 * node.half       # half-width of every child
    node.children = []
    for k in range(8):
        sign = np.array([1.0 if k & 1 else -1.0,
                         1.0 if k & 2 else -1.0,
                         1.0 if k & 4 else -1.0])
        node.children.append(Node(node.centre + quarter * sign, quarter))


# Insert body i into the subtree that starts at node.
# A leaf may hold a single body. When a second body arrives the leaf is subdivided, the
# body already living there is pushed down into its octant, and the insertion continues
# into the octant of the new body. This repeats until every body sits alone in a leaf.
# node – current node of the descent
# i – index of the body in bod_obj
# bod_obj – a list of stellar bodies represented as objects
# depth – depth of node in the tree (the root has depth 0)
def insert_bh(node, i, bod_obj, depth):
    if node.children is None:
        # Empty leaf (or the depth limit is reached): simply store the body here
        if not node.bodies or depth >= max_depth_bh:
            node.bodies.append(i)
            return

        # Occupied leaf: subdivide and push the resident body one level down
        subdivide_bh(node)
        for j in node.bodies:
            insert_bh(node.children[octant_bh(node, bod_obj[j].r)], j, bod_obj, depth + 1)
        node.bodies = []

    # Internal node: descend into the octant that contains body i
    insert_bh(node.children[octant_bh(node, bod_obj[i].r)], i, bod_obj, depth + 1)


# Compute the total mass and the centre of mass of every node (post-order traversal).
# Leaf: M = sum of the masses of its bodies, R = their mass-weighted mean position.
# Internal node: the same sums taken over the 8 children, M = sum_k M_k, R = sum_k M_k R_k / M.
# node – root of the subtree
# bod_obj – a list of stellar bodies represented as objects
def moments_bh(node, bod_obj):
    if node.children is None:
        for j in node.bodies:
            node.m += bod_obj[j].m
            node.com += bod_obj[j].m * bod_obj[j].r
    else:
        for child in node.children:
            moments_bh(child, bod_obj)
            node.m += child.m
            node.com += child.m * child.com

    if node.m > 0.0:
        node.com /= node.m      # numerator above is sum m r, divide by the mass to obtain R


# Build the complete octree for the current positions of all bodies.
# The root is the smallest cube that encloses every body, enlarged by a tiny margin so that
# no body lies exactly on its outer boundary.
# bod_obj – a list of stellar bodies represented as objects
def build_tree_bh(bod_obj):
    positions = np.array([obj.r for obj in bod_obj])
    lower, upper = positions.min(axis=0), positions.max(axis=0)
    centre = 0.5 * (lower + upper)
    half = 0.5 * np.max(upper - lower) * (1.0 + 1e-6) + 1e-12    # +1e-12 guards a single body

    root = Node(centre, half)
    for i in range(len(bod_obj)):
        insert_bh(root, i, bod_obj, 0)
    moments_bh(root, bod_obj)

    return root


# Check whether the displacement r lies inside the cube of a node
def contains_bh(node, r):
    return np.all(np.abs(r - node.centre) <= node.half)


# Acceleration of body i due to all the mass inside the subtree that starts at node.
# Opening criterion: a node of side s = 2*half seen from distance d = |r_i - R| is accepted
# as a single point mass at its centre of mass if s / d < theta, otherwise it is opened and
# its children are visited. A node that contains body i itself is always opened (see the
# explanation below). Leaves are summed exactly, which is Eq. (4) restricted to the leaf.
# node – current node of the tree walk
# i – index of the body that feels the force
# bod_obj – a list of stellar bodies represented as objects
# G – the constant of gravitational attraction
# e – softening factor (same Plummer softening as in acc)
# theta – opening angle
def node_acc_bh(node, i, bod_obj, G, e, theta):
    a_i = np.zeros(3)
    if node.m == 0.0:
        return a_i      # empty cube: no contribution

    r_i = bod_obj[i].r

    # Leaf: exact summation over the bodies stored in it (skipping self-interaction)
    if node.children is None:
        for j in node.bodies:
            if j == i:
                continue
            r_ij = r_i - bod_obj[j].r
            denominator = (np.dot(r_ij, r_ij) + e**2)**1.5
            a_i += -G * bod_obj[j].m * r_ij / denominator
        return a_i

    # Internal node: accept it as one pseudo-body or open it
    r_iN = r_i - node.com       # displacement from the centre of mass of the node to body i
    dist = np.sqrt(np.dot(r_iN, r_iN))
    s = 2.0 * node.half         # side length of the cube

    # s / d < theta is written without a division, and the (more expensive) containment test is
    # only evaluated for nodes that already pass the opening criterion
    if s < theta * dist and not contains_bh(node, r_i):
        denominator = (dist**2 + e**2)**1.5
        return -G * node.m * r_iN / denominator

    for child in node.children:
        a_i += node_acc_bh(child, i, bod_obj, G, e, theta)

    return a_i


# Compute Gravitational Acceleration of All Bodies with the Barnes-Hut approximation.
# A drop-in replacement for acc: same arguments and same output (a list of 3-vectors).
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects
# e – softening factor
# theta – opening angle; theta = 0 opens every node and reproduces direct summation,
#         larger values are faster but less accurate (0.5 - 1.0 is the usual range)
def acc_bh(G, bod_obj, e, theta=0.5):
    root = build_tree_bh(bod_obj)       # the tree is rebuilt at every force evaluation, since the bodies move
    return [node_acc_bh(root, i, bod_obj, G, e, theta) for i in range(len(bod_obj))]


# Barnes-Hut force routine with a fixed opening angle.
# acc_bh takes theta as a fourth argument, whereas the integrators call acc_func(G, bod_obj, e). This
# returns a function with the three-argument signature and theta fixed, which can be handed to any
# integrator. Creating it through a function call also binds the current value of theta, so it is safe
# to use inside a loop over several opening angles.
# theta – opening angle
def acc_bh_theta(theta):
    return lambda G, bod_obj, e: acc_bh(G, bod_obj, e, theta)


# Relative force error of every body, |a_i(BH) - a_i(direct)| / |a_i(direct)|.
# Used to validate the tree (theta = 0 must give zero up to rounding) and to measure its accuracy.
# a_tree – accelerations from acc_bh
# a_direct – accelerations from acc for the same configuration
def force_error_bh(a_tree, a_direct):
    return np.array([np.linalg.norm(at - ad) / np.linalg.norm(ad) for at, ad in zip(a_tree, a_direct)])


# Size of an octree: number of nodes, number of non-empty leaves and maximum depth.
# The depth shows how finely the tree has to resolve the closest pair of bodies: a tight binary
# forces many subdivisions before its two members end up in separate leaves.
# node – root of the subtree
# depth – depth of node in the tree (the root has depth 0)
def tree_stats_bh(node, depth=0):
    if node.children is None:
        return 1, int(len(node.bodies) > 0), depth

    n_nodes, n_leaves, max_depth = 1, 0, depth
    for child in node.children:
        c_nodes, c_leaves, c_depth = tree_stats_bh(child, depth + 1)
        n_nodes += c_nodes
        n_leaves += c_leaves
        max_depth = max(max_depth, c_depth)

    return n_nodes, n_leaves, max_depth

## Barnes-Hut Octree Explanation
The cost of a gravitational N-body simulation is dominated by the force evaluation. Eq. (4) sums the contribution of every body $j\neq i$ to the acceleration of body $i$, which requires $N(N-1)$ pair evaluations per call, and the 4th-order Velocity Verlet calls it three times per timestep. For the large $N$ of cosmological applications this quadratic cost is prohibitive.

The Barnes-Hut algorithm (Barnes & Hut, 1986) rests on a simple physical observation: a compact group of bodies that is far away from body $i$ pulls on it almost exactly like a single body carrying the total mass of the group and located at its centre of mass. The algorithm organises the bodies into a hierarchy of cubic cells, an *octree*, and decides for every cell whether it is distant enough to be replaced by such a pseudo-body or whether it has to be examined in more detail.

**Notation.** In addition to the notation of the Velocity Verlet explanation, $c$ denotes a cell (a node of the tree), $\hat{c}$ its geometric centre, $s$ its side length (`2 * half` in the code), $M_c$ the total mass it contains and $\hat{R}_c$ its centre of mass. $d$ is the distance from body $i$ to $\hat{R}_c$ and $\theta$ is the opening angle.

### Spatial Decomposition (Octree Construction)
The root cell is the smallest cube that encloses all bodies (`build_tree_bh`). Every cell can be split into eight equal octants of half the side length (`subdivide_bh`). The octant of a cell that contains a displacement $\hat{r}=(x,y,z)$ is identified by three binary comparisons with the centre $\hat{c}=(c_x,c_y,c_z)$ (`octant_bh`, Eq. (16)):

$$k=[x\geq c_x]+2\,[y\geq c_y]+4\,[z\geq c_z],\qquad k\in\{0,\dots,7\} \tag{16}$$

where $[\,\cdot\,]$ equals 1 if the condition holds and 0 otherwise. The centre of child $k$ is displaced from the parent centre by a quarter of the parent side length along each axis (Eq. (17)),

$$\hat{c}_k=\hat{c}+\frac{s}{4}\,(\sigma_x,\sigma_y,\sigma_z),\qquad \sigma\in\{-1,+1\} \tag{17}$$

with each sign $\sigma$ taken from the corresponding bit of $k$.

The bodies are inserted one by one (`insert_bh`). A body descends from the root into the octant that contains it until it reaches an empty leaf, where it is stored. When it reaches a leaf that is already occupied, that leaf is subdivided and both bodies are pushed further down, until each of them occupies a leaf of its own. For a roughly uniform distribution the depth of the tree grows as $\log_8 N$, so the construction costs $\mathcal{O}(N\log N)$. Two bodies at (almost) identical positions cannot be separated by any number of subdivisions, so a maximum depth (`max_depth_bh`) is imposed, beyond which a leaf may hold more than one body.

### Mass Moments of the Cells
Every cell is described by its total mass and its centre of mass (`moments_bh`, Eq. (18)):

$$M_c=\sum_{j\in c}m_j,\qquad \hat{R}_c=\frac{1}{M_c}\sum_{j\in c}m_j\,\hat{r}_j \tag{18}$$

These are computed from the bottom of the tree upwards: a leaf sums over its bodies directly, while an internal cell combines the moments of its eight children, $M_c=\sum_k M_k$ and $\hat{R}_c=\sum_k M_k\hat{R}_k/M_c$. Every body therefore contributes to one cell per level, and the whole pass costs $\mathcal{O}(N\log N)$.

### Opening Criterion and Tree Walk
The acceleration of body $i$ is obtained by walking the tree from the root (`node_acc_bh`). For each cell the ratio of its size to its distance from body $i$ is compared with the opening angle,

$$\frac{s}{d}<\theta,\qquad d=\lvert\hat{r}_i-\hat{R}_c\rvert \tag{19}$$

If the criterion of Eq. (19) holds, the cell subtends a small angle as seen from body $i$ and its entire content is replaced by a single softened point mass at its centre of mass: the contribution $\hat{a}_i^{(c)}$ of the cell is one term of the sum in Eq. (4), with $m_j\to M_c$ and $\hat{r}_j\to\hat{R}_c$. If the criterion fails, the cell is *opened* and the same test is applied to each of its children. When a leaf is reached, its bodies are summed exactly as in Eq. (4). The total acceleration is the sum of all accepted contributions (`acc_bh`).

A cell that contains body $i$ itself is always opened, regardless of the criterion. Without this rule a large cell whose centre of mass happens to lie far from body $i$ could pass the test although body $i$ sits inside it, which produces large force errors (Salmon & Warren, 1994). The rule also guarantees that body $i$ never interacts with itself, since it can only be met inside its own leaf, where the term $j=i$ is skipped.

The opening angle controls the trade-off between accuracy and cost:
- $\theta=0$: no cell is ever accepted, every cell is opened down to its leaves, and the algorithm reproduces direct summation exactly (up to the order of floating-point additions). This limit is used in Experiment 2 to validate the implementation.
- $\theta\approx0.5$–$1$: the range commonly used in practice.
- larger $\theta$: faster, but increasingly inaccurate.

### Accuracy and Cost
Because the expansion is taken about the centre of mass, the dipole term of the multipole expansion of a cell vanishes identically. The leading error of the point-mass approximation is therefore the quadrupole term (Salmon & Warren, 1994), and the relative error of a single accepted cell scales as

$$\frac{\lvert\delta\hat{a}_i^{(c)}\rvert}{\lvert\hat{a}_i^{(c)}\rvert}\sim\mathcal{O}\!\left(\left(\frac{s}{d}\right)^2\right)\lesssim\mathcal{O}(\theta^2) \tag{20}$$

so, by Eq. (20), halving $\theta$ reduces the force error roughly fourfold. The number of cells accepted by a single body grows only logarithmically with $N$ (approximately as $\theta^{-3}\log N$), hence one complete force evaluation costs $\mathcal{O}(N\log N)$ instead of $\mathcal{O}(N^2)$ (Barnes & Hut, 1986). The tree has to be rebuilt at every force evaluation, since the bodies move, but this is also an $\mathcal{O}(N\log N)$ operation.

In this pure-Python prototype the bookkeeping of the tree outweighs the saving for the small $N$ that can be simulated, so `acc_bh` is not yet faster than `acc`. The asymptotic advantage becomes decisive only for large $N$ in the compiled C++ implementation.

### Consequences for the Conserved Quantities
The approximation does not only introduce a small force error; it also breaks the symmetries on which the conservation laws of the previous section rely.

**Linear momentum.** In direct summation the force of $j$ on $i$ is exactly opposite to the force of $i$ on $j$ (Newton's third law), so the internal forces cancel in pairs and $\hat{P}$ (Eq. (13)) is conserved. In the tree, body $i$ may see body $j$ as part of a distant cell, while body $j$ sees body $i$ individually or as part of a different cell. The two forces are then no longer equal and opposite, and $\hat{P}$ drifts (Springel, 2005).

**Angular momentum.** For the same reason, and because an accepted cell pulls towards $\hat{R}_c$ rather than along the individual lines joining the bodies, the internal torques no longer cancel and $\hat{L}$ (Eq. (14)) is not exactly conserved either.

**Energy.** Velocity Verlet and its Yoshida composition are symplectic for forces that derive from a potential (Hairer, Lubich & Wanner, 2006), $\hat{a}_i=-\frac{1}{m_i}\nabla_{\hat{r}_i}V$. The tree force is not the gradient of the exact potential $V$ of the Hamiltonian, nor of any single smooth potential, since the tree is rebuilt at every evaluation and the force changes discontinuously whenever a cell switches between being accepted and being opened. The bounded energy oscillation that characterises a symplectic integrator is therefore no longer guaranteed, and $H_{rel}$ (Eq. (15)), which is always evaluated with the exact $V$ (Eq. (11)), is expected to be larger and to grow with $\theta$.

Measuring how strongly these three properties degrade for a given $\theta$ is precisely the purpose of Experiment 2.

### One Force Evaluation in the Code
1. Find the bounding cube of all bodies and insert them one by one into the octree (`build_tree_bh`, `insert_bh`, `subdivide_bh`, `octant_bh`).
2. Compute $M_c$ and $\hat{R}_c$ for every cell from the leaves upwards (`moments_bh`).
3. For every body $i$, walk the tree from the root, accepting cells that satisfy $s/d<\theta$ and do not contain body $i$, and opening all others (`node_acc_bh`, `contains_bh`).
4. Return the list of accelerations in the same format as `acc` (`acc_bh`), so that `run_vv_4(..., acc_func=...)` can use it without modification.

In [ ]:
"""
Experiment 2: Barnes-Hut octree against direct summation in a Plummer star cluster (30/10/26).

Purpose:
    Experiment 1 validated the Velocity Verlet 4th order integrator on a problem with a
    known exact answer. This experiment keeps the integrator fixed and changes only the
    force routine, in order to isolate the effect of the Barnes-Hut approximation on the
    reliability of the simulation. The same initial conditions are integrated with the
    same timestep three times:
        - direct summation, acc (the reference, exact up to rounding)
        - Barnes-Hut, acc_bh with theta = 0.5 (the usual accuracy setting)
        - Barnes-Hut, acc_bh with theta = 1.0 (a deliberately coarse setting)
    Any difference between the runs is therefore caused by the force approximation alone.

Setup:
    - N equal-mass bodies drawn from a Plummer sphere (Plummer 1911), the standard
      self-gravitating test model of N-body dynamics, sampled with the method of
      Aarseth, Henon & Wielen (1974)
    - Henon units: G = 1, total mass M = 1, total energy E = -1/4, which fixes the
      Plummer scale length to a = 3*pi/16
    - the cluster is moved to its centre-of-mass frame and its velocities are rescaled to
      exact virial equilibrium (2T = -V), so it should neither collapse nor expand
    - Plummer softening is switched ON: the bodies are randomly placed, so close encounters
      do happen, and the softening keeps the force finite (Eq. 4)
    - the system is fully three-dimensional, so all eight octants of the tree are used

Time control:
    Identical to Experiment 1: t_total is a whole number of crossing times, dt = t_total / n_steps,
    and one snapshot is stored every steps_per_frame = n_steps / n_frames steps.

Conservation tracking:
    H, P and L are evaluated at every stored frame with the EXACT expressions of the
    conservation cell (direct pair sums), never with the tree. The Barnes-Hut runs are thus
    judged against the true Hamiltonian, which is the quantity that matters physically.

Expected outcome:
    - direct summation: P and L conserved to rounding error, H_rel small and bounded
    - Barnes-Hut: P and L no longer exact (tree forces violate Newton's third law), and
      H_rel larger, growing with theta (tree forces are not the gradient of the exact V)
    - individual trajectories of the three runs agree at first and then separate, because
      the N-body problem is chaotic and amplifies any small force difference exponentially
"""


# The Plummer sampler init_plummer, the virial rescaling virialise and copy_bodies are defined in the
# helper cells (initial conditions and helpers), as they are shared with the JSON runtime.


# Integrate one copy of the initial conditions and record trajectories and conservation errors.
# The loop is the same as in Experiment 1, wrapped in a function so that it can be repeated
# for every force routine. Only the time spent inside run_vv_4 is measured as the runtime.
# bod_init – initial conditions (left untouched)
# acc_func – force routine handed to run_vv_4
def simulate(bod_init, G, e, dt, n_frames, steps_per_frame, acc_func):
    bod_obj = copy_bodies(bod_init)

    H_0, P_0, L_0 = conservation_state(bod_obj, G, e)
    P_sc, L_sc = p_scale(bod_obj), l_scale(bod_obj)

    trajectories = [[obj.r.copy()] for obj in bod_obj]
    h_err, p_err, l_err = [0.0], [0.0], [0.0]
    runtime = 0.0

    for _ in range(n_frames):
        clock = time.perf_counter()
        run_vv_4(bod_obj, G, e, dt, steps_per_frame, acc_func)
        runtime += time.perf_counter() - clock

        for k, obj in enumerate(bod_obj):
            trajectories[k].append(obj.r.copy())

        H_n, P_n, L_n = conservation_state(bod_obj, G, e)
        h_err.append(h_rel(H_0, H_n))
        p_err.append(p_rel(P_0, P_n, P_sc))
        l_err.append(l_rel(L_0, L_n, L_sc))

    return {"traj": np.array(trajectories), "h": np.array(h_err), "p": np.array(p_err),
            "l": np.array(l_err), "time": runtime}


# Experiment parameters (Henon units: G = 1, M = 1, E = -1/4)
G_nat = 1.0     # gravitational constant
M_tot = 1.0     # total mass of the cluster
a_pl = 3.0 * np.pi / 16.0       # Plummer scale length giving E = -1/4 in Henon units
N_bodies = 64       # number of bodies (kept small: the prototype is pure Python)

# Softening: a few per cent of the scale length. It must be large enough to tame close
# encounters at the chosen dt, and small compared with the mean separation of the bodies
# (about 0.4 length units here) so that the cluster still behaves like point masses.
e_soft_2 = 0.05

# The crossing time t_cross = 2 sqrt(2) (constants cell) is the time a typical body needs to cross
# the cluster in Henon units. Two crossing times are long enough for the bodies to complete roughly
# one orbit and for the runs to separate, while keeping the pure-Python runtime modest.
n_crossings = 2     # simulated time in crossing times
n_steps_2 = 500     # total number of timesteps (dictates dt)
n_frames_2 = 250    # number of stored snapshots (independent of dt)

t_total_2 = n_crossings * t_cross
dt_2 = t_total_2 / n_steps_2
assert n_steps_2 % n_frames_2 == 0, "n_steps_2 must be divisible by n_frames_2"
steps_per_frame_2 = n_steps_2 // n_frames_2

# Force routines under test. acc_bh takes the opening angle as an extra argument, so acc_bh_theta
# fixes theta and leaves the (G, bod_obj, e) signature that run_vv_4 expects.
theta_values = [0.5, 1.0]
force_routines = {"Direct summation": acc}
for th in theta_values:
    force_routines[f"Barnes-Hut theta = {th}"] = acc_bh_theta(th)

# Initial conditions (re-seeded here, so the cluster is the same regardless of which cells ran before)
rnd.seed(seed)
cluster = init_plummer(N_bodies, G_nat, M_tot, a_pl)
virialise(cluster, G_nat, e_soft_2)

print(f"N = {N_bodies}, t_total = {t_total_2:.4f} ({n_crossings} crossing times), "
      f"n_steps = {n_steps_2}, dt = {dt_2:.3e}, softening = {e_soft_2}")
H_init = h_abs(t_abs(cluster), v_abs(cluster, G_nat, e_soft_2))
print(f"H_0 = {H_init:.5f} (-0.25 for an infinite Plummer sphere; the difference is sampling noise of finite N), "
      f"virial ratio 2T/|V| = {2 * t_abs(cluster) / abs(v_abs(cluster, G_nat, e_soft_2)):.3f}")

# Force accuracy at t = 0
# Before integrating, the tree accelerations are compared body by body with direct summation:
# delta_i = |a_i(BH) - a_i(direct)| / |a_i(direct)|. theta = 0 must reproduce direct summation
# to rounding error, which validates the tree construction and the walk.
a_direct = acc(G_nat, cluster, e_soft_2)
print("\nForce accuracy at t = 0 (relative error per body):")
for th in [0.0] + theta_values:
    a_tree = acc_bh(G_nat, cluster, e_soft_2, th)
    delta = force_error_bh(a_tree, a_direct)
    print(f"  theta = {th:.1f}: mean = {delta.mean():.2e}, max = {delta.max():.2e}")

# Simulations (the same initial conditions, three force routines)
print("\nRunning simulations (this takes a few minutes in pure Python)...")
results = {}
for label, func in force_routines.items():
    results[label] = simulate(cluster, G_nat, e_soft_2, dt_2, n_frames_2, steps_per_frame_2, func)
    res = results[label]
    print(f"  {label:<24} runtime = {res['time']:6.1f} s, "
          f"max H_rel = {res['h'].max():.2e}, max P_rel = {res['p'].max():.2e}, max L_rel = {res['l'].max():.2e}")

t_frames_2 = np.arange(n_frames_2 + 1) * steps_per_frame_2 * dt_2 / t_cross     # time in crossing times

# Divergence from the reference: mean distance between the same body in a tree run and in the
# direct run at the end of the simulation, in units of the Plummer scale length.
ref_label = "Direct summation"
for label, res in results.items():
    if label != ref_label:
        sep = np.linalg.norm(res["traj"][:, -1] - results[ref_label]["traj"][:, -1], axis=1).mean()
        print(f"  mean final separation from the direct run ({label}) = {sep / a_pl:.2e} a")

# Plots: 3D and 2D trajectories + the three relative conservation errors
# The trajectories shown are those of the theta = 0.5 run. With 64 bodies a full plot would be
# unreadable, so a handful of bodies that stay inside the cluster (r < 2.5 a for the whole run)
# are drawn in colour, while the final positions of all bodies are shown as grey dots.
show_label = f"Barnes-Hut theta = {theta_values[0]}"
traj_show = results[show_label]["traj"]
traj_ref = results[ref_label]["traj"]

n_show = 6
bound = [k for k in range(N_bodies) if np.linalg.norm(traj_show[k], axis=1).max() < 2.5 * a_pl]
highlight = rnd.sample(bound, min(n_show, len(bound)))
colours_2 = plt.cm.tab10(np.arange(len(highlight)))
lim_2 = 1.1 * np.abs(traj_show[highlight]).max()

fig = plt.figure(figsize=(16, 11))
grid = fig.add_gridspec(2, 6)

# 3D view of the highlighted orbits
ax3d = fig.add_subplot(grid[0, 0:3], projection="3d")
final = traj_show[:, -1]
inside = np.all(np.abs(final) < lim_2, axis=1)      # 3D axes do not clip, so far-away bodies are left out
ax3d.scatter(*final[inside].T, color="grey", s=6, alpha=0.5, label="all bodies (final)")
for c, k in enumerate(highlight):
    x, y, z = traj_show[k].T
    ax3d.plot(x, y, z, color=colours_2[c], lw=1.2)
    ax3d.scatter(x[0], y[0], z[0], color=colours_2[c], marker="x", s=30)
    ax3d.scatter(x[-1], y[-1], z[-1], color=colours_2[c], s=30)
ax3d.set_xlim(-lim_2, lim_2)
ax3d.set_ylim(-lim_2, lim_2)
ax3d.set_zlim(-lim_2, lim_2)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title(rf"Orbits in 3D, Barnes-Hut $\theta$ = {theta_values[0]} (x = start, o = end)")
ax3d.legend(fontsize=8, loc="upper left")

# 2D projection onto the x-y plane, with the direct-summation orbit of the same body dashed.
# The two curves start on top of each other and separate as the chaotic dynamics amplify the
# small force differences: this is the visual counterpart of the final-separation printout.
ax2d = fig.add_subplot(grid[0, 3:6])
ax2d.scatter(final[:, 0], final[:, 1], color="grey", s=6, alpha=0.5)
for c, k in enumerate(highlight):
    ax2d.plot(traj_show[k, :, 0], traj_show[k, :, 1], color=colours_2[c], lw=1.4)
    ax2d.plot(traj_ref[k, :, 0], traj_ref[k, :, 1], color=colours_2[c], lw=0.9, ls="--")
    ax2d.scatter(traj_show[k, 0, 0], traj_show[k, 0, 1], color=colours_2[c], marker="x", s=30)
    ax2d.scatter(traj_show[k, -1, 0], traj_show[k, -1, 1], color=colours_2[c], s=30)
ax2d.plot([], [], color="black", lw=1.4, label=rf"Barnes-Hut $\theta$ = {theta_values[0]}")
ax2d.plot([], [], color="black", lw=0.9, ls="--", label="Direct summation")
ax2d.set_xlim(-lim_2, lim_2)
ax2d.set_ylim(-lim_2, lim_2)
ax2d.set_aspect("equal")
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Orbits projected onto the x-y plane")
ax2d.legend(fontsize=8, loc="upper left")
ax2d.grid(alpha=0.3)

# Relative conservation errors of all three runs on common axes
floor = 1e-18
run_colours = {ref_label: "black"}
for th, colour in zip(theta_values, ["tab:blue", "tab:orange"]):
    run_colours[f"Barnes-Hut theta = {th}"] = colour

panels_2 = [
    ("h", r"$H_{rel}$", "Relative Hamiltonian error"),
    ("p", r"$P_{rel}$ (relative to $\sum_i m_i|\hat{v}_i(0)|$)", "Relative linear momentum error"),
    ("l", r"$L_{rel}$", "Relative angular momentum error"),
]
for idx, (key, ylabel, title) in enumerate(panels_2):
    ax = fig.add_subplot(grid[1, 2 * idx:2 * idx + 2])
    for label, res in results.items():
        ax.semilogy(t_frames_2, np.maximum(res[key], floor), color=run_colours[label], lw=1.1,
                    label=label.replace("theta", r"$\theta$"))     # LaTeX theta in the legend
    ax.set_xlabel("time (crossing times)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)

fig.suptitle(f"Plummer cluster, N = {N_bodies}: {n_crossings} crossing times, {n_steps_2} steps, "
             f"dt = {dt_2:.2e} (Velocity Verlet 4th order, direct summation vs Barnes-Hut)")
plt.tight_layout()
plt.show()

# Creating Leapfrog Integrator

## About Leapfrog

The leapfrog method is a second-order symplectic integrator and a close relative of the Velocity Verlet integrator; in the kick-drift-kick (KDK) form applied here the two are, in fact, algebraically identical (see the explanation below). The name refers to the staggered structure of the scheme: the velocities are evaluated half a timestep out of phase with the displacements, so that the two quantities "leap" over one another in turn. The implementation reuses the helper functions and the gravitational acceleration of Eq. (1) to Eq. (4).

Within each timestep the algorithm proceeds in four stages:
- **Kick:** advance the velocities by half a timestep with the current acceleration $\hat{a}_n$
- **Drift:** advance the displacements by a full timestep with the half-step velocities
- **Force evaluation:** compute the new accelerations $\hat{a}_{n+1}$ at the new displacements
- **Kick:** advance the velocities by the remaining half timestep with $\hat{a}_{n+1}$

In [ ]:
"""
Leapfrog Second-Order Implementation.
Consists of the displacement, velocity and runtime computations.

Detailed algorithmic solution will be provided later down below in the description.
"""

# Compute initial kick by half a timestep
# obj – individual object with accessible properties
# t – current timestep
# Theoretically, the initial half-velocity will be assigned to the object and retrieved later
def veli_l_2(obj, t):
    return obj.v + obj.a * (0.5 * t)

# Compute Displacement applying first kick
# obj – individual object with accessible properties
# t – current timestep
def displ_l_2(obj, t):
    return obj.r + obj.v * t

# Compute final kick by the second half-step
# obj – individual object with accessible properties
# t – current timestep
def velf_l_2(obj, t):
    return obj.v + obj.a * (0.5 * t)

# Run Leapfrog
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_l_2(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Kick: half-step velocity of ALL bodies with the old acceleration a_n
        for obj in bod_obj:
            obj.v = veli_l_2(obj, dt)

        # Drift: move ALL bodies by a full step with the half-step velocity
        for obj in bod_obj:
            obj.r = displ_l_2(obj, dt)

        # Re-evaluate the forces once for the whole system (new acceleration a_{n+1})
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

        # Kick: complete the velocity update with the new acceleration a_{n+1}
        for obj in bod_obj:
            obj.v = velf_l_2(obj, dt)

## Leapfrog 2nd Order Explanation
The leapfrog integrator is a second-order method in its own right and therefore requires no Yoshida composition. It advances the system with the kick-drift-kick (KDK) sequence introduced above, in which a *kick* changes only the velocities and a *drift* changes only the displacements.

**Notation.** In this section the subscript $n$ denotes the timestep, so that $\hat{r}_n$, $\hat{v}_n$ and $\hat{a}_n$ are the displacement, velocity and acceleration of a body at the time $t_n=n\,\delta{t}$, while $\hat{v}_{n+1/2}$ is its velocity at the intermediate time $t_n+\delta{t}/2$. The body index $i$ is omitted for readability; every equation applies to all bodies simultaneously.

### Update Equations
**First Kick** (`veli_l_2`, Eq. (21))

$$\hat{v}_{n+1/2}=\hat{v}_n+\frac{1}{2}\,\hat{a}_n\,\delta{t} \tag{21}$$

**Drift** (`displ_l_2`, Eq. (22))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_{n+1/2}\,\delta{t} \tag{22}$$

The new acceleration $\hat{a}_{n+1}$ is then obtained from Eq. (4) at the displacements $\hat{r}_{n+1}$, and the velocity update is completed by a second half kick:

**Second Kick** (`velf_l_2`, Eq. (23))

$$\hat{v}_{n+1}=\hat{v}_{n+1/2}+\frac{1}{2}\,\hat{a}_{n+1}\,\delta{t} \tag{23}$$

The functions `veli_l_2` and `velf_l_2` contain the same expression. They differ only in the acceleration stored on the body at the moment they are called, which is $\hat{a}_n$ for the first kick and $\hat{a}_{n+1}$ for the second.

### Kick and Drift Operators
The step can be written compactly in terms of two elementary maps: the kick $K(h)$, which sets $\hat{v}\leftarrow\hat{v}+\hat{a}(\hat{r})\,h$ at fixed $\hat{r}$, and the drift $D(h)$, which sets $\hat{r}\leftarrow\hat{r}+\hat{v}\,h$ at fixed $\hat{v}$. One leapfrog step is then (Eq. (24))

$$S_2^{\mathrm{L}}(\delta{t})=K\!\left(\tfrac{\delta{t}}{2}\right)\circ D(\delta{t})\circ K\!\left(\tfrac{\delta{t}}{2}\right) \tag{24}$$

where the maps are applied from right to left (the sequence is symmetric, so it reads the same in either direction). The drift is the exact solution of the equations of motion for the kinetic part $T$ of the Hamiltonian of Eq. (12) alone, and the kick is the exact solution for the potential part $V$ alone. The exact flow of any Hamiltonian is symplectic, and a composition of symplectic maps is again symplectic; hence the leapfrog step is symplectic. Because the sequence is symmetric, the step is also time-reversible, $S_2^{\mathrm{L}}(-\delta{t})\circ S_2^{\mathrm{L}}(\delta{t})=\mathrm{id}$. Its local error is $\mathcal{O}(\delta{t}^3)$ and its global error $\mathcal{O}(\delta{t}^2)$, and, as for every symplectic method, the energy error remains bounded and oscillatory instead of drifting (Hairer, Lubich & Wanner, 2006).

### Relation to Velocity Verlet
Substituting Eq. (21) into Eq. (22) gives $\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t}+\frac{1}{2}\hat{a}_n\,\delta{t}^2$, which is Eq. (5), and adding Eq. (21) and Eq. (23) gives $\hat{v}_{n+1}=\hat{v}_n+\frac{1}{2}(\hat{a}_n+\hat{a}_{n+1})\,\delta{t}$, which is Eq. (6). The KDK leapfrog is therefore algebraically identical to the second-order Velocity Verlet step $S_2$ (Swope et al., 1982). The two differ only in the order of the floating-point operations, so that their results agree to rounding error.

### Cost and the Staggered Form
A single force evaluation is required per timestep: the acceleration $\hat{a}_{n+1}$ computed for the second kick is stored on the body and reused by the first kick of the following step. Moreover, when the velocities are not required at whole timesteps, the second kick of one step and the first kick of the next can be merged into a single full kick (Eq. (25)),

$$\hat{v}_{n+1/2}=\hat{v}_{n-1/2}+\hat{a}_n\,\delta{t} \tag{25}$$

which is the classical staggered form that gives the method its name: the displacements are defined at whole timesteps and the velocities at half timesteps, each leaping over the other. The implementation retains the synchronised KDK form, since the Hamiltonian and the momenta of Eq. (12) to Eq. (15) require displacements and velocities at the same instant.

### One Integration Step in the Code
Before the first step, `run_l_2` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Kick all velocities by half a timestep with $\hat{a}_n$ (Eq. (21)).
2. Drift all displacements by a full timestep (Eq. (22)).
3. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).
4. Kick all velocities by the remaining half timestep with $\hat{a}_{n+1}$ (Eq. (23)).

All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Forest-Ruth 4th Order Integrator (Forest and Ruth, 1990)

## About

The Forest-Ruth 4th order integrator (FR4) is an explicit symplectic integrator for separable Hamiltonians of the form $H=T(\hat{p})+V(\hat{r})$, such as Eq. (12), and therefore exhibits no long-term energy drift. It is implemented here in the form in which Forest and Ruth (1990) originally formulated it: as a sequence of four kick-drift stages, each with its own pair of coefficients, rather than as a composition of second-order steps. It is nevertheless mathematically identical to the Velocity Verlet 4th order of Eq. (7); the two differ only in the way in which they are composed (see the explanation below).

FR4 follows this pattern within each timestep:
- Stage 1: kick-drift operation giving $\hat{v}_1$ and $\hat{r}_1$, followed by the acceleration $\hat{a}_1=\hat{a}(\hat{r}_1)$
- Stage 2: kick-drift operation giving $\hat{v}_2$ and $\hat{r}_2$, followed by $\hat{a}_2=\hat{a}(\hat{r}_2)$
- Stage 3: kick-drift operation giving $\hat{v}_3$ and $\hat{r}_3$, followed by $\hat{a}_3=\hat{a}(\hat{r}_3)$
- Stage 4: final kick $\hat{v}_4=\hat{v}_3+c_4\,\delta{t}\,\hat{a}_3$. There is no drift in this stage, since $d_4=0$.

In [ ]:
"""
Forest-Ruth 4th Order Implementation (Forest & Ruth, 1990).
Consists of the coefficients, the kick (velocity) and drift (displacement) sub-steps and the runtime computation.

The integrator follows the original kick-drift recurrence of Forest and Ruth (1990) with their
coefficient set in which d_4 = 0. The detailed description of the algorithm will be provided later down below.
"""

# Real root of the cubic 48 chi^3 + 24 chi^2 - 1 = 0 (called x by Forest & Ruth, 1990)
fr_chi = (2**(1/3) + 2**(-1/3) - 1) / 6

# Kick coefficients c_1..c_4 and drift coefficients d_1..d_4 (each set sums to 1)
fr_c = (fr_chi + 0.5, -fr_chi, -fr_chi, fr_chi + 0.5)
fr_d = (2*fr_chi + 1, -4*fr_chi - 1, 2*fr_chi + 1, 0.0)

# Compute the kick of an individual body (velocity update)
# obj – individual object with accessible properties
# t – sub-step size, c_k * dt
def vel_fr_4(obj, t):
    return obj.v + obj.a * t

# Compute the drift of an individual body (displacement update)
# obj – individual object with accessible properties
# t – sub-step size, d_k * dt
def displ_fr_4(obj, t):
    return obj.r + obj.v * t

# Run Forest-Ruth (4th order)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_fr_4(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Four kick-drift stages with the coefficients (c_k, d_k)
        for c, d in zip(fr_c, fr_d):

            # Kick: update ALL velocities with the current acceleration
            for obj in bod_obj:
                obj.v = vel_fr_4(obj, c * dt)

            # Stage 4 has d_4 = 0: the bodies do not move, so the current accelerations
            # stay valid and are reused by the first kick of the next timestep
            if d == 0.0:
                continue

            # Drift: move ALL bodies with the freshly kicked velocities
            for obj in bod_obj:
                obj.r = displ_fr_4(obj, d * dt)

            # Re-evaluate the forces once for the whole system at the new displacements
            for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
                obj.a = a

## Forest-Ruth 4th Order Explanation
Forest and Ruth (1990) extended the kick-drift idea of the leapfrog method to fourth order. Instead of a single symmetric kick-drift-kick sequence, a timestep is split into four stages, each consisting of one kick and one drift of a different size. The sizes are chosen so that the error terms of the individual stages cancel up to third order.

**Notation.** As in the leapfrog explanation, the body index is omitted and every equation applies to all bodies simultaneously. The subscript $k=0,\dots,4$ now counts the stages within a single timestep: $\hat{r}_0$ and $\hat{v}_0$ are the displacement and velocity at the start of the step, $\hat{r}_4$ and $\hat{v}_4$ are those at its end, $t+\delta{t}$, and $\hat{a}_k=\hat{a}(\hat{r}_k)$ is given by Eq. (4).

### The Kick-Drift Recurrence
Each stage $k=1,\dots,4$ consists of a kick with the coefficient $c_k$ followed by a drift with the coefficient $d_k$ (`vel_fr_4` and `displ_fr_4`, Eq. (26)):

$$\hat{v}_k=\hat{v}_{k-1}+c_k\,\hat{a}_{k-1}\,\delta{t},\qquad \hat{r}_k=\hat{r}_{k-1}+d_k\,\hat{v}_k\,\delta{t} \tag{26}$$

The drift uses the velocity produced by the kick of the same stage, so the velocities of a stage must always be updated before its displacements (Forest & Ruth, 1990).

### Coefficients
The coefficients are expressed through $\chi$, the real root of the cubic of Eq. (27). Forest and Ruth (1990) denote this root $x$; the symbol $\chi$ is used here to avoid a clash with the coordinate $x$.

$$48\chi^3+24\chi^2-1=0,\qquad \chi=\frac{2^{1/3}+2^{-1/3}-1}{6}\approx 0.1756 \tag{27}$$

$$c_1=c_4=\chi+\tfrac{1}{2},\qquad c_2=c_3=-\chi,\qquad d_1=d_3=2\chi+1,\qquad d_2=-4\chi-1,\qquad d_4=0 \tag{28}$$

Numerically, $c_1\approx 0.6756$, $c_2\approx -0.1756$, $d_1\approx 1.3512$ and $d_2\approx -1.7024$. In the code they are `fr_c` and `fr_d`. Both sets sum to unity (Eq. (29)),

$$\sum_{k=1}^{4}c_k=\sum_{k=1}^{4}d_k=1 \tag{29}$$

so that the kicks and the drifts each advance the system by exactly one timestep in total; the remaining order conditions reduce to the cubic of Eq. (27). As a result the local error is $\mathcal{O}(\delta{t}^5)$ and the global error $\mathcal{O}(\delta{t}^4)$. Three of the coefficients are negative, so parts of every timestep run backwards in time, and $|d_2|>1$, so the middle drift overshoots a full timestep. These large coefficients give the method a comparatively large error constant for a fourth-order scheme, a well-known drawback that later optimised variants seek to reduce (Omelyan, Mryglod & Folk, 2002).

Since $d_4=0$, the fourth stage is a kick only, and with the kick and drift operators of Eq. (24) one full step reads (Eq. (30))

$$S_4^{\mathrm{FR}}(\delta{t})=K(c_4\delta{t})\circ D(d_3\delta{t})\circ K(c_3\delta{t})\circ D(d_2\delta{t})\circ K(c_2\delta{t})\circ D(d_1\delta{t})\circ K(c_1\delta{t}) \tag{30}$$

with the maps applied from right to left. The sequence is symmetric ($c_k=c_{5-k}$ and $d_k=d_{4-k}$), so the method is time-reversible, and as a composition of exact kicks and drifts it is symplectic.

### Relation to the Yoshida Composition
The coefficients of Eq. (28) are related to the Yoshida weights of Eq. (8) by $d_1=d_3=w_1$, $d_2=w_0$, $c_1=c_4=w_1/2$ and $c_2=c_3=(w_1+w_0)/2$. Writing each second-order step of Eq. (7) as the leapfrog step of Eq. (24) and merging the adjacent half kicks, $K(\tfrac{w_1}{2}\delta{t})\circ K(\tfrac{w_0}{2}\delta{t})=K(\tfrac{w_1+w_0}{2}\delta{t})$, reproduces Eq. (30) exactly. Since the KDK leapfrog is identical to Velocity Verlet, FR4 in this form is mathematically identical to the Velocity Verlet 4th order of `run_vv_4`: the two compute the same trajectory and differ only by rounding error. Forest and Ruth arrived at the method directly from the order conditions, while Yoshida (1990) obtained it, and integrators of arbitrary even order, by composing second-order steps. Forest and Ruth (1990) also give an equivalent coefficient set with $c_1=0$, which starts every step with a drift; it corresponds to the Yoshida composition of the drift-kick-drift leapfrog and is a distinct fourth-order method of the same cost.

### One Integration Step in the Code
Before the first step, `run_fr_4` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of four stages $k=1,\dots,4$:
1. Kick all velocities by $c_k\,\delta{t}$ with the current accelerations (Eq. (26)).
2. If $d_k\neq 0$, drift all displacements by $d_k\,\delta{t}$ (Eq. (26)).
3. If $d_k\neq 0$, re-evaluate the accelerations of all bodies at the new displacements (Eq. (4)).

In the fourth stage $d_4=0$, so the displacements are left unchanged and the accelerations $\hat{a}_3$ remain valid for the first kick of the following step. The method therefore requires three force evaluations per timestep, the same number as `run_vv_4`. All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Non-Symplectic Integrators
---

## About
Non-symplectic integration is the opposite of the previously encountered symplectic methods. They show wonders in short-term simulations, especially notable methods like RK4. However, as soon as the simulation obtains longer periods, the energy drift, linear and angular momenta, and the geometric properties of the trajectories are not conserved anymore. Hence why, the simulation runs compared between symplectic and non-symplectic methods may have different trajectories and data, even though the problem setup may be the same.

# Forward Euler

Forward Euler is the simplest of all integration methods, since it has only one order of operations and updates acceleration, displacement and velocity in this pattern. Through pure theory and practical applications this method was proven to be the simplest one to write, but even short-term property conservation is less than desired.

In [ ]:
"""
Forward Euler Implementation.
Includes the displacement, velocity and summation.
"""

# Displacement Calculation
# obj – individual object with accessible properties
# t – current timestep
# NB: this uses the OLD velocity v_n, so it has to be called before the velocities are updated.
#     With the opposite order of the two updates the method becomes the symplectic Euler method.
def displ_fe_1(obj, t):
    return obj.r + (obj.v*t)

# Velocity Calculation
# obj – individual object with accessible properties
# t – current timestep
def vel_fe_1(obj, t):
    return obj.v + (t * obj.a)

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_fe_1(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Move ALL bodies first, using the old velocity v_n
        for obj in bod_obj:
            obj.r = displ_fe_1(obj, dt)

        # Update ALL velocities with the old acceleration a_n (still stored from the previous step)
        for obj in bod_obj:
            obj.v = vel_fe_1(obj, dt)

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}),
        # so that the stored accelerations always belong to the stored displacements
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

## Forward Euler 1st Order Explanation
The forward (explicit) Euler method is the simplest of all one-step integrators: it advances the system along the tangent to the solution at the beginning of the timestep and ignores how that tangent changes during the step. It is only first-order accurate and, unlike the integrators of the previous sections, it is neither symplectic nor time-reversible. It requires a single force evaluation per timestep and is included here as the baseline of the non-symplectic comparison.

**Notation.** The notation of the leapfrog explanation is retained: the subscript $n$ denotes the timestep, $\hat{a}_n=\hat{a}(\hat{r}_n)$ is given by Eq. (4), and the body index $i$ is omitted except where a sum over the bodies is required. The kick $K(h)$ and the drift $D(h)$ are the maps of Eq. (24).

### Update Equations
The equations of motion form the first-order system (Eq. (31))

$$\dot{\hat{r}}=\hat{v},\qquad\dot{\hat{v}}=\hat{a}(\hat{r}) \tag{31}$$

which is also the starting point of the Runge-Kutta methods below. Forward Euler replaces each time derivative of Eq. (31) by a forward difference over one timestep:

**Displacement** (`displ_fe_1`, Eq. (32))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t} \tag{32}$$

**Velocity** (`vel_fe_1`, Eq. (33))

$$\hat{v}_{n+1}=\hat{v}_n+\hat{a}_n\,\delta{t} \tag{33}$$

Both right-hand sides contain only quantities at the time $t_n$, so no information from the end of the step is used. In the code this fixes the order of the operations: the displacements must be updated before the velocities, so that Eq. (32) still receives $\hat{v}_n$. If the velocities were updated first, Eq. (32) would receive $\hat{v}_{n+1}$ instead, and the scheme would become the symplectic Euler method discussed below.

### Order of Accuracy
Expanding the exact solution in a Taylor series about $t_n$ and subtracting Eq. (32) and Eq. (33) gives the error of a single step (Eq. (34)),

$$\hat{r}(t_{n+1})-\hat{r}_{n+1}=\tfrac{1}{2}\hat{a}_n\,\delta{t}^2+\mathcal{O}(\delta{t}^3),\qquad \hat{v}(t_{n+1})-\hat{v}_{n+1}=\tfrac{1}{2}\dot{\hat{a}}_n\,\delta{t}^2+\mathcal{O}(\delta{t}^3) \tag{34}$$

where the exact solution is started from $\hat{r}_n$ and $\hat{v}_n$. The displacement error is exactly the quadratic term that Eq. (5) of the Velocity Verlet method retains. The local error is therefore $\mathcal{O}(\delta{t}^2)$, and since $\mathcal{O}(1/\delta{t})$ steps are required to cover a fixed time interval, the global error is only $\mathcal{O}(\delta{t})$: halving the timestep halves the error (Hairer, Lubich & Wanner, 2006). This is the lowest order of all integrators in this notebook, while the cost per step is the same single force evaluation as for the leapfrog, which is second-order.

### Loss of the Geometric Properties
**Symplecticity.** The simplest setting in which this can be seen is a body of unit mass in one dimension with a potential $V(q)$, so that $\dot{q}=p$ and $\dot{p}=-V'(q)$. One Euler step maps $(q_n,p_n)$ to $(q_n+\delta{t}\,p_n,\ p_n-\delta{t}\,V'(q_n))$ with the Jacobian determinant (Eq. (35))

$$\det\frac{\partial(q_{n+1},p_{n+1})}{\partial(q_n,p_n)}=\det\begin{pmatrix}1&\delta{t}\\-\delta{t}\,V''(q_n)&1\end{pmatrix}=1+\delta{t}^2\,V''(q_n) \tag{35}$$

A symplectic map with one degree of freedom preserves the phase-space area and therefore has the determinant one (Hairer, Lubich & Wanner, 2006). Wherever $V''\neq0$ Eq. (35) differs from one, so the Euler step is not symplectic: the area is expanded where $V''>0$ and contracted where $V''<0$. For the harmonic oscillator $V=\tfrac{1}{2}\omega^2q^2$ the energy $E=\tfrac{1}{2}p^2+\tfrac{1}{2}\omega^2q^2$ is multiplied by the same factor in every step (Eq. (36)),

$$E_{n+1}=\left(1+\omega^2\delta{t}^2\right)E_n \tag{36}$$

so that it grows without bound instead of oscillating in a band.

**Time-reversibility.** Let $S_1^{\mathrm{FE}}(\delta{t})$ denote one Euler step. Applying a step of size $-\delta{t}$ to the result of a step of size $\delta{t}$ does not return the starting state, but yields (Eq. (37))

$$\hat{r}\;\mapsto\;\hat{r}-\delta{t}^2\,\hat{a}(\hat{r}),\qquad \hat{v}\;\mapsto\;\hat{v}+\delta{t}\left[\hat{a}(\hat{r})-\hat{a}(\hat{r}+\delta{t}\,\hat{v})\right] \tag{37}$$

so that $S_1^{\mathrm{FE}}(-\delta{t})\circ S_1^{\mathrm{FE}}(\delta{t})\neq\mathrm{id}$.

### Conserved Quantities
**Linear momentum.** Summing Eq. (33) over all bodies with the masses as weights gives (Eq. (38))

$$\hat{P}_{n+1}=\hat{P}_n+\delta{t}\sum_{i=1}^{N}m_i\,\hat{a}_{i,n}=\hat{P}_n \tag{38}$$

because the pairwise forces in Eq. (4) are equal and opposite (Newton's third law), so that $\sum_i m_i\hat{a}_i=0$. Forward Euler therefore conserves $\hat{P}$ of Eq. (13) to rounding error, as long as the force routine respects this symmetry; the Barnes-Hut approximation does not (see the Barnes-Hut explanation). The argument does not depend on the details of the Euler step: any integrator whose velocity update adds a linear combination of accelerations produced by Eq. (4), at whatever configurations they are evaluated, conserves $\hat{P}$ in the same way. This is used for the Runge-Kutta methods below.

**Angular momentum.** Inserting Eq. (32) and Eq. (33) into Eq. (14) and using $\hat{v}\times\hat{v}=0$ gives (Eq. (39))

$$\hat{L}_{n+1}=\hat{L}_n+\delta{t}\sum_{i=1}^{N}m_i\,\hat{r}_{i,n}\times\hat{a}_{i,n}+\delta{t}^2\sum_{i=1}^{N}m_i\,\hat{v}_{i,n}\times\hat{a}_{i,n}=\hat{L}_n+\delta{t}^2\sum_{i=1}^{N}m_i\,\hat{v}_{i,n}\times\hat{a}_{i,n} \tag{39}$$

The first sum vanishes, since the internal forces are central and cancel in pairs, but the second does not in general. For a body on a circular orbit, $\hat{v}\times\hat{a}$ is parallel to $\hat{L}$, so the error has the same sign in every step and accumulates. The error per step is $\mathcal{O}(\delta{t}^2)$, and the global error is again $\mathcal{O}(\delta{t})$.

**Energy.** The squared distance from the origin after one step is (Eq. (40))

$$\lvert\hat{r}_{n+1}\rvert^2=\lvert\hat{r}_n\rvert^2+2\,\delta{t}\,\hat{r}_n\cdot\hat{v}_n+\delta{t}^2\lvert\hat{v}_n\rvert^2 \tag{40}$$

For a circular orbit, or at an apsis of an elliptical one, $\hat{r}_n\perp\hat{v}_n$, so the middle term vanishes and the last one is positive: the body leaves the orbit along the tangent and ends up farther out. For a circular orbit with the angular frequency $\omega$, where $\lvert\hat{v}_n\rvert=\omega\lvert\hat{r}_n\rvert$, the radius grows by exactly $\sqrt{1+\omega^2\delta{t}^2}$ in the first step. The velocity update does the same to the speed, since $\lvert\hat{v}_{n+1}\rvert^2=\lvert\hat{v}_n\rvert^2+\delta{t}^2\lvert\hat{a}_n\rvert^2$ whenever $\hat{v}_n\cdot\hat{a}_n=0$. Both the potential and the kinetic energy of Eq. (12) rise, so the orbit spirals outward and the energy error grows systematically instead of oscillating in a bounded band, which is the behaviour described for non-symplectic methods in the section on the Hamiltonian (Hairer, Lubich & Wanner, 2006).

### Relation to the Symplectic Euler Method and the Leapfrog
The Euler step cannot be written as a composition of the kick and drift maps of Eq. (24), because both increments are evaluated at the same initial state. Composing the two maps instead, with the kick applied first, gives the symplectic Euler method (Eq. (41)),

$$S_1^{\mathrm{SE}}(\delta{t})=D(\delta{t})\circ K(\delta{t}) \tag{41}$$

that is, Eq. (33) followed by Eq. (32) with $\hat{v}_n$ replaced by the updated $\hat{v}_{n+1}$, which is `run_fe_1` with the two update loops swapped. It has the same cost and the same first order, but as a composition of exact kicks and drifts it is symplectic. The leapfrog step of Eq. (24) is, in turn, the composition of a symplectic Euler step of size $\delta{t}/2$ with its adjoint (Hairer, Lubich & Wanner, 2006). Forward Euler is retained here nonetheless, because it is the standard non-symplectic reference against which the symplectic integrators are compared.

### One Integration Step in the Code
Before the first step, `run_fe_1` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Move all bodies by a full timestep with $\hat{v}_n$ (Eq. (32)).
2. Update all velocities by a full timestep with $\hat{a}_n$ (Eq. (33)).
3. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the third stage are reused by the first velocity update of the following step, so that one force evaluation per timestep is required, and the stored accelerations always belong to the stored displacements, as in the other integrators. All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.


# Runge-Kutta 2nd Order (RK2)

## About

The Runge-Kutta methods (Runge, 1895; Kutta, 1901) raise the order of accuracy of Forward Euler by sampling the slope of the solution at several points inside a single timestep and advancing the system with a weighted average of these slopes. RK2 uses two such samples and solves the equations of motion to the second order of accuracy at the cost of two force evaluations per timestep. The variant implemented here is the explicit midpoint method: a trial half-step is taken along the tangent, and the slope found at the midpoint of that trial step is used for the full step. As with Forward Euler, the method is neither symplectic nor time-reversible, and it is therefore included as the second-order member of the non-symplectic comparison. The RK4 method presented afterwards is its fourth-order counterpart.

In [ ]:
"""
RK2 Implementation (explicit midpoint method; Runge, 1895).
Includes the stage evaluations, the displacement and velocity updates and the runtime computation.

Unlike the symplectic integrators above, the acceleration has to be evaluated at an intermediate
(trial) configuration of the whole system, which is not the state the bodies are currently in.
This is done by acc_stage, which is reused by the RK4 implementation below.
The detailed description of the algorithm will be provided later down below.
"""

# Evaluate the accelerations of ALL bodies at trial displacements
# bod_obj – a list of stellar bodies represented as objects.
# r_trial – list of trial displacement vectors, one for every body (same order as bod_obj)
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# acc_func – force routine, see run_fe_1
# NB: the displacements of the bodies are only changed temporarily. They are restored before
#     the function returns, so that the stored state of the simulation is never corrupted.
def acc_stage(bod_obj, r_trial, G, e, acc_func=acc):
    r_saved = [obj.r for obj in bod_obj]    # remember the true displacements

    try:
        for obj, r in zip(bod_obj, r_trial):
            obj.r = r
        return acc_func(G, bod_obj, e)      # forces at the trial configuration
    finally:
        for obj, r in zip(bod_obj, r_saved):
            obj.r = r                       # always restore the true displacements

# Trial displacement of an individual body: r_n + h*k_r
# obj – individual object with accessible properties
# h – (fraction of the) timestep that the stage advances by
# k_r – displacement slope of the previous stage
def stage_displ_rk(obj, h, k_r):
    return obj.r + h*k_r

# Trial velocity of an individual body: v_n + h*k_v
# obj – individual object with accessible properties
# h – (fraction of the) timestep that the stage advances by
# k_v – velocity slope (acceleration) of the previous stage
def stage_vel_rk(obj, h, k_v):
    return obj.v + h*k_v

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k2_r – displacement slope at the midpoint of the step
def displ_rk_2(obj, t, k2_r):
    return obj.r + t*k2_r

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k2_v – velocity slope (acceleration) at the midpoint of the step
def vel_rk_2(obj, t, k2_v):
    return obj.v + t*k2_v

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation, see run_fe_1
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_rk_2(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies (the slope k1_v of the first step)
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Stage 1 is free: k1_r = v_n and k1_v = a_n (already stored on every body).
        # Stage 2: take half a step with the stage 1 slopes and evaluate the slopes there (midpoint)
        r_mid = [stage_displ_rk(obj, 0.5*dt, obj.v) for obj in bod_obj]
        k2_r = [stage_vel_rk(obj, 0.5*dt, obj.a) for obj in bod_obj]
        k2_v = acc_stage(bod_obj, r_mid, G, e, acc_func)

        # Advance ALL bodies with the midpoint slopes. Everything is computed before anything is
        # assigned, so that no body uses the already updated state of another one
        r_new = [displ_rk_2(obj, dt, k) for obj, k in zip(bod_obj, k2_r)]
        v_new = [vel_rk_2(obj, dt, k) for obj, k in zip(bod_obj, k2_v)]

        for obj, r, v in zip(bod_obj, r_new, v_new):
            obj.r = r
            obj.v = v

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}).
        # They are the k1_v of the next step, so that no force is evaluated twice
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

## Runge-Kutta 2nd Order Explanation
The explicit midpoint method, which is the form of RK2 implemented here, advances the system with the slopes of two stages: the slope at the beginning of the timestep, which is used only to reach a trial midpoint, and the slope at that midpoint, which is then used for the whole step. Because the second slope is a better estimate of the average slope over the step than the first one, the method gains one order of accuracy over Forward Euler. The price is that the force has to be evaluated at a configuration that the bodies never occupy.

**Notation.** The notation of the Forward Euler explanation is retained. In addition, the slopes of the stage $s$ are written $\hat{k}_{s,r}$ for the displacement and $\hat{k}_{s,v}$ for the velocity, in accordance with the variables `k1_r`, `k1_v`, `k2_r` and `k2_v` of the code.

### Update Equations
Every stage supplies one value of the right-hand side of the first-order system of Eq. (31), introduced for Forward Euler. The first stage is the slope at the beginning of the step (Eq. (42)),

$$\hat{k}_{1,r}=\hat{v}_n,\qquad\hat{k}_{1,v}=\hat{a}_n \tag{42}$$

which is identical to the slope used by Forward Euler and is available without any further work. The second stage evaluates the slope at the midpoint $t_n+\tfrac{1}{2}\delta{t}$ of the trial half-step along the first slope (Eq. (43)),

$$\hat{k}_{2,r}=\hat{v}_n+\tfrac{1}{2}\hat{k}_{1,v}\,\delta{t},\qquad\hat{k}_{2,v}=\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\,\delta{t}\right) \tag{43}$$

where the displacement slope is the velocity at the midpoint and the velocity slope is the gravitational acceleration at the trial displacement. The second stage alone is then used to advance the system:

**Displacement** (`displ_rk_2`, Eq. (44))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{k}_{2,r}\,\delta{t} \tag{44}$$

**Velocity** (`vel_rk_2`, Eq. (45))

$$\hat{v}_{n+1}=\hat{v}_n+\hat{k}_{2,v}\,\delta{t} \tag{45}$$

The acceleration in Eq. (43) of one body depends on the displacements of all the other bodies through Eq. (4), so the trial displacements of the entire system have to exist before any $\hat{k}_{2,v}$ can be computed. This is the purpose of `acc_stage`: it temporarily places every body at its trial displacement, calls the force routine and restores the true displacements before returning, so that the stored state of the simulation is never corrupted. Since the trial configuration is passed to the force routine through the bodies themselves, any routine with the signature of `acc` can be used, including the Barnes-Hut routine `acc_bh`, which rebuilds its tree at every call.

Substituting Eq. (43) into Eq. (44) shows that the displacement update is exactly Eq. (5) of the Velocity Verlet method, while substituting it into Eq. (45) gives the velocity update (Eq. (46))

$$\hat{v}_{n+1}=\hat{v}_n+\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{v}_n\,\delta{t}\right)\delta{t} \tag{46}$$

which uses the acceleration at the midpoint of the step instead of the average of the accelerations at its ends, Eq. (6).

### Order of Accuracy
Let $\hat{y}=(\hat{r},\hat{v})$ and let $\hat{f}$ denote the right-hand side of Eq. (31), so that $\dot{\hat{y}}=\hat{f}(\hat{y})$. The Taylor series of the exact solution started from $\hat{y}_n$ is (Eq. (47))

$$\hat{y}(t_{n+1})=\hat{y}_n+\delta{t}\,\hat{f}(\hat{y}_n)+\tfrac{1}{2}\delta{t}^2\,\hat{f}'\hat{f}(\hat{y}_n)+\mathcal{O}(\delta{t}^3) \tag{47}$$

where $\hat{f}'$ is the Jacobian of $\hat{f}$. The slope of the second stage expands as (Eq. (48))

$$\hat{f}\!\left(\hat{y}_n+\tfrac{1}{2}\delta{t}\,\hat{f}(\hat{y}_n)\right)=\hat{f}(\hat{y}_n)+\tfrac{1}{2}\delta{t}\,\hat{f}'\hat{f}(\hat{y}_n)+\mathcal{O}(\delta{t}^2) \tag{48}$$

and multiplication by $\delta{t}$ reproduces Eq. (47) up to and including the term of order $\delta{t}^2$. The local error is therefore $\mathcal{O}(\delta{t}^3)$, and the global error, which accumulates over $\mathcal{O}(1/\delta{t})$ steps, is $\mathcal{O}(\delta{t}^2)$: halving the timestep reduces the error by a factor of four (Hairer, Nørsett & Wanner, 1993). The displacement can be examined explicitly. Since the displacement update is Eq. (5), the Taylor polynomial of $\hat{r}(t)$ of degree two, the error of the displacement in one step is (Eq. (49))

$$\hat{r}(t_{n+1})-\hat{r}_{n+1}=\tfrac{1}{6}\dot{\hat{a}}_n\,\delta{t}^3+\mathcal{O}(\delta{t}^4) \tag{49}$$

where $\dot{\hat{a}}_n$ is the time derivative of the acceleration along the exact trajectory. The order of the method is the same as that of the leapfrog and the Velocity Verlet method, but it requires two force evaluations per step instead of one.

### Loss of the Geometric Properties
**Symplecticity.** A Runge-Kutta method with the coefficients $a_{ij}$ and the weights $b_i$ is symplectic for all Hamiltonian systems only if (Eq. (50))

$$b_i\,a_{ij}+b_j\,a_{ji}=b_i\,b_j\qquad\text{for all } i,j \tag{50}$$

(Hairer, Lubich & Wanner, 2006). For RK2 the only non-zero coefficient is $a_{21}=\tfrac{1}{2}$, with $b_1=0$ and $b_2=1$. An explicit method has $a_{ij}=0$ for $j\geq i$, so that Eq. (50) with $i=j$ reduces to $b_i^2=0$ for every stage, which contradicts $\sum_i b_i=1$. No explicit Runge-Kutta method can therefore be symplectic, and this holds for RK2 and RK4 alike. For the harmonic oscillator of Eq. (36), with $\dot{q}=p$ and $\dot{p}=-\omega^2q$, Eq. (43) to Eq. (45) give (Eq. (51))

$$q_{n+1}=\left(1-\tfrac{1}{2}\omega^2\delta{t}^2\right)q_n+\delta{t}\,p_n,\qquad p_{n+1}=\left(1-\tfrac{1}{2}\omega^2\delta{t}^2\right)p_n-\omega^2\delta{t}\,q_n \tag{51}$$

and the cross terms cancel in $p^2+\omega^2q^2$, so that the energy is multiplied by the same factor in every step (Eq. (52)),

$$E_{n+1}=\left(1+\tfrac{1}{4}\omega^4\delta{t}^4\right)E_n \tag{52}$$

The energy grows without bound, exactly as for Forward Euler in Eq. (36), but with the fourth power of $\omega\delta{t}$ instead of the second, which is why the growth is much slower at the same timestep.

**Time-reversibility.** An explicit Runge-Kutta method is never symmetric (Hairer, Lubich & Wanner, 2006): a step of size $-\delta{t}$ applied to the result of a step of size $\delta{t}$ does not return the starting state, as was the case for Forward Euler in Eq. (37).

### Conserved Quantities
**Linear momentum.** The velocity update of Eq. (45) adds the single acceleration $\hat{k}_{2,v}$, which is the output of the same pairwise force routine as Eq. (4), only evaluated at the trial displacements, so that the forces remain equal and opposite (Newton's third law) and $\sum_i m_i\hat{k}_{2,v,i}=0$. By the argument of Eq. (38), the total momentum $\hat{P}$ of Eq. (13) is therefore conserved to rounding error, provided the force routine respects this symmetry; the Barnes-Hut approximation does not.

**Angular momentum.** The total angular momentum $\hat{L}$ of Eq. (14) is a quadratic function of the state, and Eq. (50) is precisely the condition under which a Runge-Kutta method preserves quadratic invariants. Since it cannot be satisfied by an explicit method, $\hat{L}$ is conserved only to the order of the method: the error is bounded by the local error of $\mathcal{O}(\delta{t}^3)$ per step and by the global error of $\mathcal{O}(\delta{t}^2)$, and it accumulates instead of oscillating.

**Energy.** The Hamiltonian of Eq. (12) is not conserved exactly either. Its error is of the order of the method and, as in Eq. (52), it builds up steadily with the number of steps instead of staying inside a bounded band, which is the behaviour of non-symplectic methods described in the section on the Hamiltonian (Hairer, Lubich & Wanner, 2006). The sign of the drift depends on the orbit and cannot be read off Eq. (52), which is derived for the harmonic oscillator only.

### Relation to Other Methods
As shown above, RK2 and the Velocity Verlet method update the displacements identically, Eq. (5), and differ only in the velocity update, Eq. (46) against Eq. (6), where Velocity Verlet uses the average of $\hat{a}_n$ and $\hat{a}_{n+1}$. This small difference is sufficient to destroy the symplectic property. Replacing the midpoint slope in Eq. (43) by the average of the slopes at the two ends of the step gives Heun's method, a different second-order Runge-Kutta method of the same cost. The midpoint form is retained here because of its direct relation to Eq. (5), noted above.

### One Integration Step in the Code
Before the first step, `run_rk_2` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Take the first-stage slopes from the stored state, $\hat{k}_{1,r}=\hat{v}_n$ and $\hat{k}_{1,v}=\hat{a}_n$ (Eq. (42)).
2. Form the trial midpoint displacements $\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\delta{t}$ and the displacement slopes $\hat{k}_{2,r}$ of all bodies (Eq. (43)).
3. Evaluate the accelerations $\hat{k}_{2,v}$ of all bodies at the trial displacements with `acc_stage` (Eq. (43)).
4. Advance all displacements and all velocities with the second-stage slopes (Eq. (44) and Eq. (45)). Both new lists are computed before either is assigned, so that no body uses the already updated state of another one.
5. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the last stage are the first-stage slopes $\hat{k}_{1,v}$ of the following step, so that exactly two force evaluations are required per timestep, and the stored accelerations always belong to the stored displacements, as in the other integrators.

# Runge-Kutta 4th Order (RK4)

## About

The classical fourth-order Runge-Kutta method (Kutta, 1901) extends the idea of RK2 to four stages: the slope is sampled once at the beginning of the timestep, twice at its midpoint and once at its end, and the four samples are combined with the weights $1:2:2:1$. It is the most widely used general-purpose integrator and, for a sufficiently small timestep, it is far more accurate over short simulations than any of the second-order methods of this notebook. The accuracy is bought with four force evaluations per timestep, which is the dominant cost of an N-body simulation, and it does not extend to long simulations, since RK4 is not symplectic and therefore shows a systematic energy drift.

In [ ]:
"""
RK4 Implementation (classical fourth-order Runge-Kutta method; Kutta, 1901).
Includes the displacement and velocity updates and the runtime computation.

The stage helper functions stage_displ_rk and stage_vel_rk and the trial force evaluation
acc_stage are carried over from the RK2 cell above.
The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k1_r, k2_r, k3_r, k4_r – displacement slopes of the four stages
def displ_rk_4(obj, t, k1_r, k2_r, k3_r, k4_r):
    return obj.r + (t/6.0)*(k1_r + 2.0*k2_r + 2.0*k3_r + k4_r)

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k1_v, k2_v, k3_v, k4_v – velocity slopes (accelerations) of the four stages
def vel_rk_4(obj, t, k1_v, k2_v, k3_v, k4_v):
    return obj.v + (t/6.0)*(k1_v + 2.0*k2_v + 2.0*k3_v + k4_v)

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation, see run_fe_1
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_rk_4(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies (the slope k1_v of the first step)
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Stage 1 is free: the slopes at the start of the step are k1_r = v_n and k1_v = a_n
        k1_r = [obj.v for obj in bod_obj]
        k1_v = [obj.a for obj in bod_obj]

        # Stage 2: half a step with the stage 1 slopes
        r_2 = [stage_displ_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k1_r)]
        k2_r = [stage_vel_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k1_v)]
        k2_v = acc_stage(bod_obj, r_2, G, e, acc_func)

        # Stage 3: half a step again, now with the stage 2 slopes
        r_3 = [stage_displ_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k2_r)]
        k3_r = [stage_vel_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k2_v)]
        k3_v = acc_stage(bod_obj, r_3, G, e, acc_func)

        # Stage 4: a FULL step with the stage 3 slopes
        r_4 = [stage_displ_rk(obj, dt, k) for obj, k in zip(bod_obj, k3_r)]
        k4_r = [stage_vel_rk(obj, dt, k) for obj, k in zip(bod_obj, k3_v)]
        k4_v = acc_stage(bod_obj, r_4, G, e, acc_func)

        # Advance ALL bodies with the weighted average of the four slopes (weights 1, 2, 2, 1).
        # Everything is computed before anything is assigned, so that no body uses the already
        # updated state of another one
        r_new = [displ_rk_4(obj, dt, k1_r[i], k2_r[i], k3_r[i], k4_r[i]) for i, obj in enumerate(bod_obj)]
        v_new = [vel_rk_4(obj, dt, k1_v[i], k2_v[i], k3_v[i], k4_v[i]) for i, obj in enumerate(bod_obj)]

        for obj, r, v in zip(bod_obj, r_new, v_new):
            obj.r = r
            obj.v = v

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}).
        # They are the k1_v of the next step, so that every step costs exactly four force evaluations
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

## Runge-Kutta 4th Order Explanation
The classical RK4 method advances the system with a weighted average of four slopes. Each of the three trial stages is built from the slope of the stage before it, so that the later slopes correct the errors of the earlier ones, and the weights are chosen so that the error terms of the average cancel up to the fourth power of the timestep. The cost is four force evaluations per step, each of them at a trial configuration of the whole system, which is carried out by the routine `acc_stage` introduced for RK2.

**Notation.** The notation of the RK2 explanation is retained, with the stage index now running over $s=1,\dots,4$. The right-hand side $\hat{f}$ of the first-order system is that of Eq. (31).

### Update Equations
The four stages are evaluated in turn. The first two are exactly those of RK2: the slope at the beginning of the step, which requires no work (Eq. (42)), and the slope at the midpoint of a trial half-step along it (Eq. (43)). The third stage returns to the midpoint, but now follows the improved second slope (Eq. (53)),

$$\hat{k}_{3,r}=\hat{v}_n+\tfrac{1}{2}\hat{k}_{2,v}\,\delta{t},\qquad\hat{k}_{3,v}=\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{k}_{2,r}\,\delta{t}\right) \tag{53}$$

and the fourth stage takes a trial step over the whole timestep along the third slope, in order to sample the slope at its end (Eq. (54)),

$$\hat{k}_{4,r}=\hat{v}_n+\hat{k}_{3,v}\,\delta{t},\qquad\hat{k}_{4,v}=\hat{a}\!\left(\hat{r}_n+\hat{k}_{3,r}\,\delta{t}\right) \tag{54}$$

The system is then advanced with the weighted average of the four slopes:

**Displacement** (`displ_rk_4`, Eq. (55))

$$\hat{r}_{n+1}=\hat{r}_n+\frac{\delta{t}}{6}\left(\hat{k}_{1,r}+2\hat{k}_{2,r}+2\hat{k}_{3,r}+\hat{k}_{4,r}\right) \tag{55}$$

**Velocity** (`vel_rk_4`, Eq. (56))

$$\hat{v}_{n+1}=\hat{v}_n+\frac{\delta{t}}{6}\left(\hat{k}_{1,v}+2\hat{k}_{2,v}+2\hat{k}_{3,v}+\hat{k}_{4,v}\right) \tag{56}$$

Since the displacement slopes in Eq. (43), Eq. (53) and Eq. (54) are built from the velocity slopes, the stage variables of the displacement can be eliminated (Eq. (57)),

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t}+\frac{\delta{t}^2}{6}\left(\hat{k}_{1,v}+\hat{k}_{2,v}+\hat{k}_{3,v}\right) \tag{57}$$

so that the displacement is that of a body in uniformly accelerated motion with the average of the first three accelerations, and the fourth acceleration $\hat{k}_{4,v}$ enters only the velocity. The weights $1:2:2:1$ are those of Simpson's rule: if the right-hand side of Eq. (31) depended on the time only, Eq. (55) and Eq. (56) would reduce to the quadrature (Eq. (58))

$$y_{n+1}=y_n+\frac{\delta{t}}{6}\left[f(t_n)+4f\!\left(t_n+\tfrac{1}{2}\delta{t}\right)+f(t_{n+1})\right] \tag{58}$$

in which the two midpoint slopes together carry the weight four (Hairer, Nørsett & Wanner, 1993).

### Order of Accuracy
Expanding the exact solution and the four stages in Taylor series and comparing the coefficients shows that the weights and the stage coefficients of Eq. (42), Eq. (43) and Eq. (53) to Eq. (56) satisfy all conditions for the terms up to and including $\delta{t}^4$, but not for the term of order $\delta{t}^5$ (Hairer, Nørsett & Wanner, 1993). The local error is therefore $\mathcal{O}(\delta{t}^5)$ and the global error is $\mathcal{O}(\delta{t}^4)$: halving the timestep reduces the error by a factor of sixteen. This is the highest order of the non-symplectic methods of this notebook, and the same order as that of the Velocity Verlet 4th order and the Forest-Ruth integrators. In terms of the cost, one RK4 step is as expensive as four leapfrog steps of the size $\delta{t}/4$, which is the appropriate basis for a comparison at the equal number of force evaluations.

### Loss of the Geometric Properties
**Symplecticity and time-reversibility.** RK4 is an explicit Runge-Kutta method, so that the argument following Eq. (50) applies without change: it cannot be symplectic and cannot be symmetric (Hairer, Lubich & Wanner, 2006). For a linear problem $\dot{y}=\lambda y$ one step of RK4 multiplies the solution by (Eq. (59))

$$R(z)=1+z+\frac{z^2}{2}+\frac{z^3}{6}+\frac{z^4}{24},\qquad z=\lambda\,\delta{t} \tag{59}$$

which is the Taylor polynomial of $e^z$ of degree four (Hairer, Nørsett & Wanner, 1993). The harmonic oscillator of Eq. (51) has the eigenvalues $\lambda=\pm i\omega$ and its energy is proportional to the squared modulus of the corresponding amplitude, so that the energy is multiplied in every step by (Eq. (60))

$$\left\lvert R(i\omega\delta{t})\right\rvert^2=1-\frac{(\omega\delta{t})^6}{72}+\frac{(\omega\delta{t})^8}{576} \tag{60}$$

For $\omega\delta{t}<2\sqrt{2}$ this factor is smaller than one, and the energy decays by the relative amount $(\omega\delta{t})^6/72$ per step. This is a much smaller loss than the gain of Eq. (52) at the same timestep, but it is still systematic: the energy error is not an oscillation inside a bounded band as for the symplectic methods, but a drift that accumulates with the number of steps. For $\omega\delta{t}>2\sqrt{2}$ the factor exceeds one and the numerical solution grows without bound, which sets the largest usable timestep for a system with the angular frequency $\omega$.

### Conserved Quantities
**Linear momentum.** The velocity update of Eq. (56) is a linear combination of four accelerations, each of which is the output of the pairwise force routine of Eq. (4) at some configuration and therefore satisfies $\sum_i m_i\hat{k}_{s,v,i}=0$. By the argument of Eq. (38), $\hat{P}$ of Eq. (13) is conserved to rounding error whenever the force routine respects Newton's third law, and not otherwise, as for the Barnes-Hut routine.

**Angular momentum and energy.** As for RK2, $\hat{L}$ of Eq. (14) is a quadratic invariant and is preserved only by methods that satisfy Eq. (50), and the Hamiltonian of Eq. (12) is not preserved by any Runge-Kutta method in general. Both errors are bounded by the global error of $\mathcal{O}(\delta{t}^4)$ and are therefore very small for a small timestep, but they have no mechanism to remain bounded in time. For short simulations RK4 can therefore be more accurate than the symplectic second-order methods, which is the "short-term success" of the non-symplectic methods referred to in the introduction of this section, whereas for a large number of steps the drift accumulates and eventually dominates.

### One Integration Step in the Code
Before the first step, `run_rk_4` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Take the first-stage slopes from the stored state, $\hat{k}_{1,r}=\hat{v}_n$ and $\hat{k}_{1,v}=\hat{a}_n$ (Eq. (42)).
2. Form the trial displacements $\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\delta{t}$ of all bodies, compute $\hat{k}_{2,r}$ and evaluate $\hat{k}_{2,v}$ with `acc_stage` (Eq. (43)).
3. Repeat with the second-stage slopes to obtain $\hat{k}_{3,r}$ and $\hat{k}_{3,v}$ (Eq. (53)).
4. Repeat with the third-stage slopes and the full timestep to obtain $\hat{k}_{4,r}$ and $\hat{k}_{4,v}$ (Eq. (54)).
5. Advance all displacements and all velocities with the weights $1:2:2:1$ (Eq. (55) and Eq. (56)). Both new lists are computed before either is assigned, so that no body uses the already updated state of another one.
6. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the last stage are the first-stage slopes $\hat{k}_{1,v}$ of the following step, so that exactly four force evaluations are required per timestep: three at the trial configurations and one at the new state. The trial displacements are never stored on the bodies, so that the state of the simulation is always the state at the end of the last completed step.

In [ ]:
"""
Integrator Registry (08/11/26).
All six integrators share the signature run_*(bod_obj, G, e, dt, n_steps, acc_func) and leave r and v
at the same instant on exit, so they are interchangeable. This table collects them under short keys,
which are used by the hot-swap schedule of Experiment 3 and by the JSON experiment files.
Each entry holds the run function, a readable label, the order of accuracy and whether the scheme
is symplectic.
"""

integrator_registry = {
    "L2": {"run": run_l_2, "label": "Leapfrog (2nd)", "order": 2, "symplectic": True},
    "FR4": {"run": run_fr_4, "label": "Forest-Ruth (4th)", "order": 4, "symplectic": True},
    "VV4": {"run": run_vv_4, "label": "Velocity Verlet (4th)", "order": 4, "symplectic": True},
    "FE1": {"run": run_fe_1, "label": "Forward Euler (1st)", "order": 1, "symplectic": False},
    "RK2": {"run": run_rk_2, "label": "Runge-Kutta (2nd)", "order": 2, "symplectic": False},
    "RK4": {"run": run_rk_4, "label": "Runge-Kutta (4th)", "order": 4, "symplectic": False},
}

In [ ]:
"""
Experiment 3: integrator hot-swap in a randomised 20-body system with the Barnes-Hut octree (05/11/26).

Purpose:
    In order to test the theoretical baselines of the algorithms described earlier, it is
    beneficial to design an experiment that applies both the tree recursion and the whole
    range of integrators at once, so that their performance can be compared at a glance.
    Experiment 1 fixed the force routine and validated a single integrator; Experiment 2 fixed
    the integrator and varied the force routine. This experiment fixes the force routine
    (Barnes-Hut, acc_bh) and the system, and hot-swaps the integrator during ONE continuous run:
        - the simulated time is divided into equal segments, one per integrator
        - at every segment boundary the integrator is replaced by the next one in the schedule
        - the state (r, v) left behind by one integrator is the starting state of the next one
    The same schedule is then repeated with direct summation (acc), so that the error caused
    by each integrator can be separated from the error caused by the tree.

Why a hot-swap is possible (and exact):
    All six integrators are one-step methods with the identical signature
    run_*(bod_obj, G, e, dt, n_steps, acc_func), and all of them advance the bodies in place.
    At the end of every call the displacement r_n and the velocity v_n refer to the SAME instant
    t_n (the half-step velocity of the leapfrog and the sub-steps of the Yoshida, Forest-Ruth and
    Runge-Kutta schemes are all completed inside the step). The state at a segment boundary is
    therefore an ordinary phase-space point (r, v), and no conversion is needed when the next
    integrator takes over. Every run_* also recomputes a = acc_func(r) on entry, which depends on
    the positions only, so the acceleration that the new integrator starts from is consistent
    with the state it receives. This is verified below with an identity-swap test.

Setup:
    - N = 20 bodies with RANDOMISED initial conditions (re-seeded with the notebook seed):
        masses uniform in [1 - delta, 1 + delta], normalised to the total mass M = 1
        displacements uniform inside a sphere of radius R
        velocities with isotropic random directions and random speeds
      The cluster is moved to its centre-of-mass frame and its velocities are rescaled to a
      chosen virial ratio Q = T / |V| (Q = 0.5 is equilibrium, Q < 0.5 makes it collapse).
    - Henon-like units: G = 1, M = 1. A uniform sphere of radius R has V = -3 G M^2 / (5 R), so in
      virial equilibrium E = V / 2 = -3 / (10 R), and R = 6/5 gives E = -1/4 as in Experiment 2
      (up to the sampling noise of 20 bodies and the softening)
    - unequal masses make the tree nodes' centres of mass differ from their geometric centres,
      which is a stricter test of moments_bh than the equal-mass cluster of Experiment 2
    - Plummer softening is switched ON (Eq. 4): 20 bodies form a very collisional system, in which
      close encounters and temporary binaries are frequent
    - the system is fully three-dimensional, so all eight octants of the tree are used

Time control:
    t_total = n_crossings * t_cross with the Henon crossing time t_cross = 2 sqrt(2), dt = t_total / n_steps.
    The run is split into len(schedule) segments of steps_per_segment steps each, and every segment
    into frames_per_segment frames of steps_per_frame steps. All divisions must be exact, so that every
    integrator covers exactly the same simulated time with exactly the same dt.
    NB: the timestep is the same for all integrators, but the cost per step is not (1 to 4 force
    evaluations), so the cost of every segment is measured as well.

Conservation tracking:
    H, P and L are evaluated with the EXACT expressions of the conservation cell (never with the
    tree) at every frame. Two kinds of error are reported:
        - the global errors H_rel, P_rel and L_rel relative to t = 0 (as in Experiments 1 and 2)
        - the per-segment change |H_end - H_start| / |H_0| of every segment, measured from the state
          the integrator received. A segment therefore is not blamed for the errors of the ones before it.

Tree diagnostics:
    The accuracy of the tree forces is not fixed: it depends on the configuration of the bodies,
    which keeps changing. The relative force error of acc_bh against acc is therefore measured at
    every stored frame of the Barnes-Hut run, together with the depth and the size of the octree.

Expected outcome:
    - the identity-swap test returns exactly zero, so the swap itself introduces no error
    - symplectic segments (L2, FR4, VV4): bounded, oscillating H error with no systematic drift
    - non-symplectic segments (RK2, RK4, FE1): systematic drift of H, worst for Forward Euler
    - direct summation: P conserved to rounding error in EVERY segment, whatever the integrator, because
      P is a linear invariant and every scheme only combines forces that obey Newton's third law.
      L (a quadratic invariant, r x v) is conserved to rounding error only by the symplectic kick-drift
      schemes; the Runge-Kutta and Euler segments change it, so the L_rel curve jumps at those segments
    - Barnes-Hut: P and L no longer exact, and an additional H error of roughly the same size in every
      segment, since the tree forces are not the exact gradient of V. This tree error floor hides the
      differences between the accurate integrators, which only direct summation reveals
    - the Barnes-Hut and direct runs separate within the first segments (chaos, see Experiment 2), so from
      then on the two force routines are compared on statistically similar, not identical, configurations
"""


# Body, copy_bodies and the generator init_random_sphere are defined in the helper cells (helpers and
# initial conditions), tree_stats_bh in the Barnes-Hut cell and the integrator table integrator_registry
# in its own cell above, as they are shared with the other experiments and the JSON runtime.


# A force routine that counts how many times it has been called.
# It is handed to the integrators in place of acc_func and behaves exactly like the routine it wraps,
# so that the cost of every segment can be measured in force evaluations, which (unlike wall-clock
# time) does not depend on the computer or on anything else running at the same time.
class CountedForce:
    def __init__(self, func):
        self.func = func        # the wrapped force routine, acc or acc_bh
        self.calls = 0          # number of force evaluations so far

    def __call__(self, G, bod_obj, e):
        self.calls += 1
        return self.func(G, bod_obj, e)


# Integrate one copy of the initial conditions while hot-swapping the integrator.
# The loop is the one of Experiments 1 and 2, nested inside a loop over the segments of the schedule.
# bod_init – initial conditions (left untouched)
# G – the constant of gravitational attraction
# e – softening factor
# dt – timestep (the same for every segment)
# schedule – list of keys of integrator_registry, in the order in which they take over
# frames_per_segment – number of stored snapshots in every segment
# steps_per_frame – integration steps between two stored snapshots
# acc_func – force routine used by every integrator of the run
def simulate_hot_swap(bod_init, G, e, dt, schedule, frames_per_segment, steps_per_frame, acc_func):
    bod_obj = copy_bodies(bod_init)
    force = CountedForce(acc_func)

    # References at t = 0 for the global relative errors
    H_0, P_0, L_0 = conservation_state(bod_obj, G, e)
    P_sc, L_sc = p_scale(bod_obj), l_scale(bod_obj)

    trajectories = [[obj.r.copy()] for obj in bod_obj]
    h_err, p_err, l_err = [0.0], [0.0], [0.0]
    segments = []       # one summary dictionary per segment

    for key in schedule:
        run = integrator_registry[key]["run"]     # the integrator that takes over at this boundary

        # State handed over at the boundary: the per-segment errors are measured from here
        H_start, P_start, L_start = conservation_state(bod_obj, G, e)
        calls_start = force.calls
        runtime = 0.0
        h_seg_max = 0.0     # largest |H - H_start| / |H_0| reached inside the segment

        for _ in range(frames_per_segment):
            # The integrator advances the SAME list of bodies in place: this is the hot-swap.
            # Only the time spent inside the integrator is measured as the runtime.
            clock = time.perf_counter()
            run(bod_obj, G, e, dt, steps_per_frame, force)
            runtime += time.perf_counter() - clock

            for k, obj in enumerate(bod_obj):
                trajectories[k].append(obj.r.copy())

            H_n, P_n, L_n = conservation_state(bod_obj, G, e)
            h_err.append(h_rel(H_0, H_n))
            p_err.append(p_rel(P_0, P_n, P_sc))
            l_err.append(l_rel(L_0, L_n, L_sc))
            h_seg_max = max(h_seg_max, abs(H_n - H_start) / abs(H_0))

        # Force evaluations of the segment. Every run_* call evaluates the forces once on entry, so
        # (calls - frames) / steps is the cost of the scheme itself per step: 1 for L2 and FE1, 2 for
        # RK2, 3 for VV4 and FR4 and 4 for RK4. It doubles as a check of the implementations.
        n_calls = force.calls - calls_start
        steps = frames_per_segment * steps_per_frame
        segments.append({
            "key": key,
            "dH": abs(H_n - H_start) / abs(H_0),        # net change of H over the segment
            "dH_max": h_seg_max,        # largest excursion of H inside the segment
            "dP": np.linalg.norm(P_n - P_start) / P_sc,     # change of P over the segment
            "dL": np.linalg.norm(L_n - L_start) / L_sc,     # change of L over the segment
            "calls": n_calls,
            "evals_per_step": (n_calls - frames_per_segment) / steps,
            "time": runtime,
        })

    return {"traj": np.array(trajectories), "h": np.array(h_err), "p": np.array(p_err),
            "l": np.array(l_err), "segments": segments, "masses": [obj.m for obj in bod_obj]}


# Experiment parameters (Henon-like units: G = 1, M = 1, E ~ -1/4)
G_nat = 1.0     # gravitational constant
M_tot = 1.0     # total mass of the system
N_bodies_3 = 20     # number of bodies
R_sphere = 1.2      # radius of the initial sphere, R = 6/5 gives E = -1/4 in virial equilibrium (see above)
mass_spread = 0.5       # masses uniform in [0.5, 1.5] before normalisation
q_virial = 0.5      # virial ratio T / |V|; 0.5 is equilibrium

# Softening: the mean separation of 20 bodies in a sphere of radius 1.2 is roughly
# R / N^(1/3) ~ 0.44, so e = 0.05 is small compared with it (the bodies still behave like point
# masses) but large enough to keep the closest encounters resolvable with a fixed timestep.
e_soft_3 = 0.05

# Opening angle of the tree. theta = 0.5 is the usual accuracy setting used in Experiment 2.
theta_3 = 0.5

# Hot-swap schedule. Integrators of the same order are placed next to each other, symplectic first,
# so that each pair can be compared directly: L2 against RK2, then FR4 and VV4 against RK4.
# Forward Euler comes last: it is by far the least accurate scheme, and placing it earlier would
# distort the configuration that every later integrator receives. The list can be reordered freely
# (or an integrator repeated), as long as its length divides n_frames_3.
schedule_3 = ["L2", "RK2", "FR4", "RK4", "VV4", "FE1"]

# Time discretisation: one crossing time per integrator (t_cross is defined in the constants cell)
n_crossings_3 = len(schedule_3)     # simulated time in crossing times
n_steps_3 = 3000        # total number of timesteps (dictates dt)
n_frames_3 = 600        # number of stored snapshots (independent of dt)

t_total_3 = n_crossings_3 * t_cross
dt_3 = t_total_3 / n_steps_3

# Every segment must hold a whole number of frames, and every frame a whole number of steps
assert n_frames_3 % len(schedule_3) == 0, "n_frames_3 must be divisible by the number of segments"
assert n_steps_3 % n_frames_3 == 0, "n_steps_3 must be divisible by n_frames_3"
frames_per_segment_3 = n_frames_3 // len(schedule_3)
steps_per_frame_3 = n_steps_3 // n_frames_3
steps_per_segment_3 = frames_per_segment_3 * steps_per_frame_3

# Barnes-Hut force routine with the opening angle fixed (as in Experiment 2)
acc_bh_3 = acc_bh_theta(theta_3)

# Initial conditions (re-seeded here, so the system is the same regardless of which cells ran before)
rnd.seed(seed)
system_3 = init_random_sphere(N_bodies_3, G_nat, M_tot, R_sphere, e_soft_3, mass_spread, q_virial)

masses_3 = np.array([b.m for b in system_3])
T_init, V_init = t_abs(system_3), v_abs(system_3, G_nat, e_soft_3)
print(f"N = {N_bodies_3}, masses in [{masses_3.min():.4f}, {masses_3.max():.4f}] (sum = {masses_3.sum():.6f})")
print(f"t_total = {t_total_3:.4f} ({n_crossings_3} crossing times), n_steps = {n_steps_3}, dt = {dt_3:.3e}, "
      f"softening = {e_soft_3}, theta = {theta_3}")
print(f"{len(schedule_3)} segments of {steps_per_segment_3} steps ({frames_per_segment_3} frames of "
      f"{steps_per_frame_3} steps): {' -> '.join(schedule_3)}")
print(f"H_0 = {T_init + V_init:.5f} (-0.25 expected up to sampling noise), virial ratio T/|V| = {T_init / abs(V_init):.3f}")

# Tree check at t = 0
# As in Experiment 2: theta = 0 must reproduce direct summation to rounding error, and theta_3
# gives the typical force error of the runs below. The tree statistics show its size for 20 bodies.
a_direct = acc(G_nat, system_3, e_soft_3)
print("\nForce accuracy at t = 0 (relative error per body):")
for th in [0.0, theta_3, 1.0]:
    a_tree = acc_bh(G_nat, system_3, e_soft_3, th)
    delta = force_error_bh(a_tree, a_direct)
    print(f"  theta = {th:.1f}: mean = {delta.mean():.2e}, max = {delta.max():.2e}")
n_nodes, n_leaves, depth = tree_stats_bh(build_tree_bh(system_3))
print(f"Octree at t = 0: {n_nodes} nodes, {n_leaves} occupied leaves, maximum depth {depth}")

# Hot-swap validation: the identity swap
# If the swap itself were lossy, splitting a run of one integrator into several consecutive calls would
# change the result. For every integrator, 12 steps are taken once in a single call and once as four
# calls of 3 steps (i.e. the integrator is "swapped for itself" three times). The two final states must
# agree to the last bit, because the only thing that happens at a swap, the re-evaluation of a from r,
# reproduces exactly the acceleration that the integrator would have used anyway.
print("\nIdentity-swap test (one call of 12 steps vs four calls of 3 steps):")
for key, entry in integrator_registry.items():
    single = copy_bodies(system_3)
    entry["run"](single, G_nat, e_soft_3, dt_3, 12, acc)
    split = copy_bodies(system_3)
    for _ in range(4):
        entry["run"](split, G_nat, e_soft_3, dt_3, 3, acc)
    diff = max(max(np.abs(a.r - b.r).max(), np.abs(a.v - b.v).max()) for a, b in zip(single, split))
    print(f"  {key:<4} max |difference| in r and v = {diff:.1e}")

# Simulations: the same initial conditions and schedule, two force routines
# NB: for only 20 bodies the tree is SLOWER than direct summation: building it and walking it costs more
# than the 380 pair interactions it replaces. Its O(N log N) advantage only appears for much larger N, so
# here the tree is tested for its reliability, not for its speed.
print("\nRunning the hot-swap simulations...")
force_routines_3 = {"Barnes-Hut": acc_bh_3, "Direct summation": acc}
results_3 = {}
for label, func in force_routines_3.items():
    results_3[label] = simulate_hot_swap(system_3, G_nat, e_soft_3, dt_3, schedule_3,
                                         frames_per_segment_3, steps_per_frame_3, func)
    res = results_3[label]
    print(f"  {label:<17} runtime = {sum(s['time'] for s in res['segments']):6.1f} s, "
          f"max H_rel = {res['h'].max():.2e}, max P_rel = {res['p'].max():.2e}, max L_rel = {res['l'].max():.2e}")

# Per-segment summary table: one row per integrator for each force routine.
# dH is the net change of H over the segment and dH_max its largest excursion, both relative to |H_0|.
# For a symplectic scheme dH_max is typically larger than dH (the error oscillates and partly returns),
# for a non-symplectic one the two are close (the error accumulates in one direction).
for label, res in results_3.items():
    print(f"\nPer-segment summary, {label}:")
    print(f"  {'segment':<24}{'sympl.':<8}{'evals/step':>11}{'time (s)':>10}{'dH':>11}{'dH_max':>11}{'dP':>11}{'dL':>11}")
    for seg in res["segments"]:
        entry = integrator_registry[seg["key"]]
        print(f"  {entry['label']:<24}{'yes' if entry['symplectic'] else 'no':<8}{seg['evals_per_step']:>11.2f}"
              f"{seg['time']:>10.2f}{seg['dH']:>11.2e}{seg['dH_max']:>11.2e}{seg['dP']:>11.2e}{seg['dL']:>11.2e}")

# Tree accuracy along the trajectory
# The stored displacements of the Barnes-Hut run are loaded back into a set of bodies frame by frame, and
# the tree accelerations are compared with direct summation in exactly the configuration the run was in.
# Accelerations depend on the positions and masses only, so the stored trajectory is all that is needed.
traj_bh = results_3["Barnes-Hut"]["traj"]
probe = copy_bodies(system_3)
force_err_mean, force_err_max, tree_depth = [], [], []
for f in range(n_frames_3 + 1):
    for k, obj in enumerate(probe):
        obj.r = traj_bh[k, f]
    a_ex = acc(G_nat, probe, e_soft_3)
    a_tr = acc_bh_3(G_nat, probe, e_soft_3)
    delta = force_error_bh(a_tr, a_ex)
    force_err_mean.append(delta.mean())
    force_err_max.append(delta.max())
    tree_depth.append(tree_stats_bh(build_tree_bh(probe))[2])
force_err_mean, force_err_max, tree_depth = np.array(force_err_mean), np.array(force_err_max), np.array(tree_depth)
print(f"\nTree force error along the run: mean = {force_err_mean.mean():.2e}, worst body = {force_err_max.max():.2e}; "
      f"tree depth between {tree_depth.min()} and {tree_depth.max()}")

# Divergence of the two runs: mean distance between the same body in the tree run and in the direct run
# at the end of every segment. The N-body problem is chaotic, so this grows even though the per-step
# force difference stays small (see Experiment 2).
traj_dir = results_3["Direct summation"]["traj"]
sep = np.linalg.norm(traj_bh - traj_dir, axis=2).mean(axis=0)       # shape: (n_frames + 1,)
ends = [(s + 1) * frames_per_segment_3 for s in range(len(schedule_3))]
print("Mean separation between the Barnes-Hut and direct runs at the end of each segment (in units of R): "
      + ", ".join(f"{key} {sep[f] / R_sphere:.1e}" for key, f in zip(schedule_3, ends)))

# Escapers: small-N systems eject bodies through close encounters. A body further than 3R from the
# centre of mass at the end of the run is counted as (likely) ejected and is left out of the 3D plot.
final_r = np.linalg.norm(traj_bh[:, -1], axis=1)
escapers = np.where(final_r > 3.0 * R_sphere)[0]
print(f"Bodies further than 3R from the centre at the end of the Barnes-Hut run: {len(escapers)}")

# Plots
# Row 1: the Barnes-Hut orbits in 3D and projected onto the x-y plane, coloured by SEGMENT (i.e. by the
#        integrator that was in charge at that time) rather than by body, so the hot-swap is visible.
# Row 2: global relative errors H_rel, P_rel, L_rel of both runs, with the segments shaded.
# Row 3: tree force error along the run, per-segment change of H, and per-segment cost.
t_frames_3 = np.arange(n_frames_3 + 1) * steps_per_frame_3 * dt_3 / t_cross     # time in crossing times
seg_colours = plt.cm.tab10(np.arange(len(schedule_3)))
seg_bounds = [s * frames_per_segment_3 for s in range(len(schedule_3) + 1)]     # frame index of each boundary
run_styles = {"Barnes-Hut": ("tab:blue", "-"), "Direct summation": ("black", "--")}

fig = plt.figure(figsize=(16, 16))
grid = fig.add_gridspec(3, 6)

stay = [k for k in range(N_bodies_3) if k not in escapers]      # bodies drawn in the 3D view
lim_3 = 1.1 * np.abs(traj_bh[stay]).max()

# 3D view
ax3d = fig.add_subplot(grid[0, 0:3], projection="3d")
for k in stay:
    for s in range(len(schedule_3)):
        x, y, z = traj_bh[k, seg_bounds[s]:seg_bounds[s + 1] + 1].T     # +1 joins neighbouring segments
        ax3d.plot(x, y, z, color=seg_colours[s], lw=0.9)
    ax3d.scatter(*traj_bh[k, 0], color="black", marker="x", s=15)       # starting position
    ax3d.scatter(*traj_bh[k, -1], color="black", s=15)      # final position
ax3d.set_xlim(-lim_3, lim_3)
ax3d.set_ylim(-lim_3, lim_3)
ax3d.set_zlim(-lim_3, lim_3)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title(rf"Orbits in 3D, Barnes-Hut $\theta$ = {theta_3} (x = start, o = end)")

# 2D view (2D axes clip, so every body can be drawn here, escapers included)
ax2d = fig.add_subplot(grid[0, 3:6])
for k in range(N_bodies_3):
    for s in range(len(schedule_3)):
        seg = traj_bh[k, seg_bounds[s]:seg_bounds[s + 1] + 1]
        ax2d.plot(seg[:, 0], seg[:, 1], color=seg_colours[s], lw=0.9)
    ax2d.scatter(traj_bh[k, 0, 0], traj_bh[k, 0, 1], color="black", marker="x", s=20, zorder=3)
    ax2d.scatter(traj_bh[k, -1, 0], traj_bh[k, -1, 1], color="black", s=20, zorder=3)
for s, key in enumerate(schedule_3):
    ax2d.plot([], [], color=seg_colours[s], lw=2, label=integrator_registry[key]["label"])        # legend entries only
ax2d.set_xlim(-lim_3, lim_3)
ax2d.set_ylim(-lim_3, lim_3)
ax2d.set_aspect("equal")
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Orbits projected onto the x-y plane, coloured by integrator")
ax2d.legend(fontsize=8, loc="upper left")
ax2d.grid(alpha=0.3)


# Shade the segments of a time axis and label each of them with its integrator key
def shade_segments_3(ax):
    for s, key in enumerate(schedule_3):
        ax.axvspan(t_frames_3[seg_bounds[s]], t_frames_3[seg_bounds[s + 1]], color=seg_colours[s], alpha=0.12, lw=0)
        ax.text(0.5 * (t_frames_3[seg_bounds[s]] + t_frames_3[seg_bounds[s + 1]]), 1.01, key,
                transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=8)


# Global relative conservation errors of both runs
floor = 1e-18       # keeps exact zeros drawable on a log axis
panels_3 = [
    ("h", r"$H_{rel}$", "Relative Hamiltonian error"),
    ("p", r"$P_{rel}$ (relative to $\sum_i m_i|\hat{v}_i(0)|$)", "Relative linear momentum error"),
    ("l", r"$L_{rel}$", "Relative angular momentum error"),
]
for idx, (key, ylabel, title) in enumerate(panels_3):
    ax = fig.add_subplot(grid[1, 2 * idx:2 * idx + 2])
    shade_segments_3(ax)
    for label, res in results_3.items():
        colour, style = run_styles[label]
        ax.semilogy(t_frames_3, np.maximum(res[key], floor), color=colour, ls=style, lw=1.1, label=label)
    ax.set_xlabel("time (crossing times)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=14)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)

# Tree force error along the Barnes-Hut run, with the depth of the octree on a second axis.
# Peaks in the depth mark close pairs of bodies, which the tree has to resolve with many subdivisions.
ax_f = fig.add_subplot(grid[2, 0:2])
shade_segments_3(ax_f)
ax_f.semilogy(t_frames_3, force_err_mean, color="tab:blue", lw=1.1, label="mean over bodies")
ax_f.semilogy(t_frames_3, force_err_max, color="tab:red", lw=0.8, label="worst body")
ax_f.set_xlabel("time (crossing times)")
ax_f.set_ylabel(r"$|\hat{a}_{BH}-\hat{a}_{direct}|/|\hat{a}_{direct}|$")
ax_f.set_title(rf"Tree force error along the run ($\theta$ = {theta_3})", pad=14)
ax_f.grid(alpha=0.3, which="both")
ax_depth = ax_f.twinx()
ax_depth.step(t_frames_3, tree_depth, color="grey", lw=0.7, where="mid", alpha=0.7)
ax_depth.plot([], [], color="grey", lw=0.7, label="tree depth (right axis)")
ax_depth.set_ylabel("maximum tree depth")
lines = ax_f.get_legend_handles_labels()
lines_d = ax_depth.get_legend_handles_labels()
ax_depth.legend(lines[0] + lines_d[0], lines[1] + lines_d[1], fontsize=8, loc="upper left")

# Per-segment change of H, side by side for both force routines (log scale).
# This is the clearest single summary of the experiment: every bar is the error that ONE integrator
# added during its own crossing time, independently of the integrators before it.
x_pos = np.arange(len(schedule_3))
width = 0.38
ax_dh = fig.add_subplot(grid[2, 2:4])
for offset, label in zip([-0.5 * width, 0.5 * width], results_3):
    colour, _ = run_styles[label]
    ax_dh.bar(x_pos + offset, [max(s["dH"], floor) for s in results_3[label]["segments"]], width,
              color=colour, alpha=0.8, label=label)
ax_dh.set_yscale("log")
ax_dh.set_xticks(x_pos, schedule_3)
for tick, s in zip(ax_dh.get_xticklabels(), range(len(schedule_3))):
    tick.set_color(seg_colours[s])
ax_dh.set_ylabel(r"$|H_{end}-H_{start}|/|H_0|$")
ax_dh.set_title("Change of H during each segment")
ax_dh.grid(alpha=0.3, which="major", axis="y")
ax_dh.legend(fontsize=8)

# Per-segment cost: force evaluations per step of the scheme (bars) and wall-clock time of the Barnes-Hut
# run (dots, right axis). Accuracy per unit cost is what decides which integrator is worth using.
ax_c = fig.add_subplot(grid[2, 4:6])
segs_bh = results_3["Barnes-Hut"]["segments"]
ax_c.bar(x_pos, [s["evals_per_step"] for s in segs_bh], 0.6, color=seg_colours, alpha=0.8)
ax_c.set_xticks(x_pos, schedule_3)
ax_c.set_ylabel("force evaluations per step")
ax_c.set_title("Cost of each segment")
ax_c.grid(alpha=0.3, axis="y")
ax_t = ax_c.twinx()
ax_t.plot(x_pos, [s["time"] for s in segs_bh], "o", color="black", label="runtime, Barnes-Hut (right axis)")
ax_t.plot(x_pos, [s["time"] for s in results_3["Direct summation"]["segments"]], "s", color="grey",
          mfc="none", label="runtime, direct (right axis)")
ax_t.set_ylabel("runtime per segment (s)")
ax_t.set_ylim(bottom=0)
ax_t.legend(fontsize=8, loc="upper left")

fig.suptitle(f"Randomised {N_bodies_3}-body system: integrator hot-swap {' -> '.join(schedule_3)}, "
             f"{n_steps_3} steps, dt = {dt_3:.2e} (Barnes-Hut vs direct summation)")
plt.tight_layout(rect=(0, 0, 1, 0.98))        # leave room for the overall title
plt.show()

# Automation: Loggers, Plotters and JSON Experiments
---
Experiments 1 to 3 each contain their own copy of the recording loop and of the plotting code. The cells below factor this code out into three reusable components, so that a new experiment no longer requires a new cell:

- **Logger:** records the trajectories, the absolute Hamiltonian, the absolute linear and angular momenta and their relative errors into a single `np.array` table with one row per stored frame.
- **Plotter:** draws the orbits in 3D and in any of the three 2D projections, together with the relative Hamiltonian drift and the relative linear and angular momentum errors. Every panel can be included in or excluded from the output individually.
- **JSON loader and runtime:** an experiment is described in a JSON file (integrator or hot-swap schedule, force routine, time discretisation, initial conditions and plot selection), which `run_experiment` loads, checks, simulates, logs and plots.

The complete description of the file format and of the pipeline is given in `README.md`; example experiment files are kept in the `experiments/` folder.

In [ ]:
"""
Loggers and Plotters (08/11/26).
The following describes the design of automated plotter and logger functions in order to
further automate the design process of the tool. Experiments 1 to 3 each carry their own copy of
the same recording loop and of the same plotting code; the functions below factor that code out,
so that every future experiment (in particular the JSON-driven experiments of the runtime cells
below) records and displays its results in exactly the same way.

Logger:
    The log of one run is a plain dictionary. Its central entry, log["table"], is a single 2D NumPy
    array with ONE ROW PER STORED FRAME and the following columns:
        t | H | P_abs | L_abs | H_rel | P_rel | L_rel | x_1 | y_1 | z_1 | ... | x_N | y_N | z_N
    where
        t – simulated time of the frame
        H – absolute Hamiltonian (total energy), T + V
        P_abs, L_abs – absolute values (norms) of the total linear and angular momentum vectors
        H_rel, P_rel, L_rel – relative errors with respect to t = 0 (h_rel, p_rel and l_rel of the
                              conservation cell), computed at the same moment as the row is logged
        x_k, y_k, z_k – displacement of body k
    A flat table of this kind maps one-to-one onto a CSV file (one row per line, one header per
    column), so saving the results later will only require a single np.savetxt call.
    The table is preallocated for the whole run, so logging a frame never resizes the array.

Plotter:
    plot_log draws any combination of the following panels from a finished log:
        - the orbits in 3D
        - one or more 2D projections of the orbits (x-y, x-z, y-z)
        - the relative Hamiltonian drift H_rel
        - the relative linear and angular momentum errors P_rel and L_rel
    Every panel can be switched on or off individually (this is how the "plots" block of an
    experiment JSON file includes or excludes a plot from the output).
"""


# Names of the scalar columns of the log table, in the order in which they are stored.
# The displacement columns x_k, y_k, z_k of every body follow after them.
log_scalar_columns = ["t", "H", "P_abs", "L_abs", "H_rel", "P_rel", "L_rel"]
n_scalar_columns = len(log_scalar_columns)


# Create the log of a run and record the initial state (row 0) in it.
# G – the constant of gravitational attraction
# e – softening factor. It is stored in the log and reused for every later frame, so that H is
#     always evaluated with the same softening as the forces (see v_abs).
# bod_obj – a list of stellar bodies represented as objects (the state at the start of the run).
# n_frames – number of frames that will be logged AFTER the initial state; the table therefore
#            has n_frames + 1 rows.
# t – simulated time of the initial state (normally zero)
def init_log(G, e, bod_obj, n_frames, t=0.0):
    N = len(bod_obj)

    # Conservation references
    # The state at t = 0 is the reference for every relative error. If P_0 or L_0 is exactly
    # zero (as in a centre-of-mass frame), p_rel and l_rel fall back on the characteristic size
    # of the individual momenta, p_scale and l_scale, which are therefore stored as well.
    H_0, P_0, L_0 = conservation_state(bod_obj, G, e)

    # A relative energy error is undefined for H_0 = 0 (e.g. an exactly parabolic system), since
    # h_rel divides by |H_0|. It is caught here rather than producing a column of infinities.
    if abs(H_0) == 0.0:
        raise ValueError("H_0 = 0: the relative Hamiltonian error |H_n - H_0| / |H_0| is undefined")

    # Column names: the scalar columns followed by x_k, y_k, z_k of every body (bodies numbered from 1,
    # as in the plot legends of the experiments)
    columns = list(log_scalar_columns)
    for k in range(N):
        columns += [f"x_{k + 1}", f"y_{k + 1}", f"z_{k + 1}"]

    log = {
        "table": np.full((n_frames + 1, len(columns)), np.nan),     # NaN marks rows not yet written
        "columns": columns,     # header of the table (one name per column)
        "index": {name: c for c, name in enumerate(columns)},       # column name -> column number
        "n_rows": 0,        # number of rows written so far
        "N": N,     # number of bodies
        "G": G,
        "e": e,
        "H_0": H_0,     # references of the relative errors
        "P_0": P_0,
        "L_0": L_0,
        "P_scale": p_scale(bod_obj),
        "L_scale": l_scale(bod_obj),
        "segments": [],     # (first row, label) of every integrator segment, see log_segment
    }

    log_state(log, bod_obj, t)      # row 0: relative errors are zero by definition

    return log


# Record the current state of the system as the next row of the table.
# log – the log created by init_log
# bod_obj – a list of stellar bodies represented as objects (the state to be logged)
# t – simulated time of the state
def log_state(log, bod_obj, t):
    row = log["n_rows"]
    if row >= log["table"].shape[0]:
        raise IndexError(f"the log is full: it was created for {log['table'].shape[0]} rows")
    if len(bod_obj) != log["N"]:
        raise ValueError(f"the log was created for {log['N']} bodies, but {len(bod_obj)} were passed")

    # Absolute conserved quantities, evaluated with the EXACT pair sums of the conservation cell
    # (never with the tree), so that runs with different force routines are judged on the true H
    H_n, P_n, L_n = conservation_state(bod_obj, log["G"], log["e"])

    # Relative errors, computed alongside the absolute values and stored in the same row
    h_r = h_rel(log["H_0"], H_n)
    p_r = p_rel(log["P_0"], P_n, log["P_scale"])
    l_r = l_rel(log["L_0"], L_n, log["L_scale"])

    # Scalar part of the row
    log["table"][row, :n_scalar_columns] = [t, H_n, np.linalg.norm(P_n), np.linalg.norm(L_n), h_r, p_r, l_r]

    # Displacements: [r_1, r_2, ..., r_N] flattened into x_1, y_1, z_1, x_2, ... (body-major order)
    log["table"][row, n_scalar_columns:] = np.concatenate([obj.r for obj in bod_obj])

    log["n_rows"] += 1


# Mark the start of a new integrator segment at the NEXT row to be logged (used by the hot-swap
# schedules of the runtime). The plotter shades the segments on the conservation panels.
# log – the log created by init_log
# label – name of the integrator that takes over (e.g. "VV4")
def log_segment(log, label):
    # The segment starts at the last logged row: that row is the state handed over to the new integrator
    log["segments"].append((max(log["n_rows"] - 1, 0), label))


# The rows written so far (a view, not a copy). Every accessor below works on this part only,
# so a log can be inspected or plotted even if the run was interrupted half-way.
def log_rows(log):
    return log["table"][:log["n_rows"]]


# One column of the table by name, e.g. log_column(log, "H_rel")
# log – the log created by init_log
# name – column name (see log["columns"])
def log_column(log, name):
    return log_rows(log)[:, log["index"][name]]


# Trajectories of all bodies, reshaped from the flat table into the layout used in Experiments 1 to 3:
#   axis 0 – body index k, axis 1 – frame index, axis 2 – coordinate (x, y, z)
def log_trajectories(log):
    flat = log_rows(log)[:, n_scalar_columns:]      # shape: (rows, 3N)
    return flat.reshape(log["n_rows"], log["N"], 3).transpose(1, 0, 2)      # shape: (N, rows, 3)


# Short printed summary of a finished log (the same quantities that Experiments 1 to 3 print)
def log_summary(log):
    print(f"H_0 = {log['H_0']:.6f}, |P_0| = {np.linalg.norm(log['P_0']):.1e}, |L_0| = {np.linalg.norm(log['L_0']):.1e}")
    print(f"max H_rel = {np.nanmax(log_column(log, 'H_rel')):.3e}, "
          f"max P_rel = {np.nanmax(log_column(log, 'P_rel')):.3e}, "
          f"max L_rel = {np.nanmax(log_column(log, 'L_rel')):.3e}, "
          f"final H_rel = {log_column(log, 'H_rel')[-1]:.3e}")


# Default plot selection. Any key left out of the "plots" block of an experiment falls back on these.
#   orbit_3d – orbits in 3D
#   orbit_2d – list of planes to project the orbits onto ("xy", "xz", "yz"); [] switches 2D views off
#   h_rel, p_rel, l_rel – relative Hamiltonian drift and relative linear/angular momentum errors
#   max_bodies – largest number of orbits drawn as lines; with more bodies only a subset is drawn
#   highlight – explicit list of body indices (counted from 1) to draw, overrides max_bodies
#   log_scale – logarithmic y-axis on the conservation panels
plot_defaults = {
    "orbit_3d": True,
    "orbit_2d": ["xy"],
    "h_rel": True,
    "p_rel": True,
    "l_rel": True,
    "max_bodies": 10,
    "highlight": None,
    "log_scale": True,
}

# Coordinate indices of every projection plane
plot_planes = {"xy": (0, 1), "xz": (0, 2), "yz": (1, 2)}

# Choose which bodies are drawn as orbits.
# With few bodies all of them are drawn. With many, the plot would be unreadable (see Experiment 2),
# so the max_bodies bodies that stay closest to the origin over the whole run are chosen: they
# remain inside the plotted box, whereas escapers would stretch the axes and squash everything else.
# The choice is deterministic, so the same run always produces the same figure.
def choose_bodies_plot(traj, opts):
    N = traj.shape[0]
    if opts["highlight"] is not None:
        return [k - 1 for k in opts["highlight"]]       # the JSON numbers bodies from 1
    if N <= opts["max_bodies"]:
        return list(range(N))
    r_max = np.linalg.norm(traj, axis=2).max(axis=1)        # furthest distance reached by every body
    return sorted(np.argsort(r_max)[:opts["max_bodies"]].tolist())

# Plot a finished log.
# log – the log created by init_log and filled by log_state
# plots – plot selection (see plot_defaults); missing keys take their default values
# title – overall title of the figure
# t_unit – time unit of the x-axis of the conservation panels, in simulation units (e.g. T_period)
# t_unit_name – name of that unit, used in the axis label
# Returns the figure, or None if every panel was switched off.
def plot_log(log, plots=None, title="", t_unit=1.0, t_unit_name="simulation time units"):
    opts = {**plot_defaults, **(plots or {})}
    planes = opts["orbit_2d"] or []     # False / None / [] all mean "no 2D projections"

    # Panels of the two rows: orbit views on top, conservation errors underneath (as in Experiment 1).
    # P_ref and L_ref are |P_0| and |L_0|, or p_scale and l_scale when the initial momentum is zero
    # (see p_rel and l_rel in the conservation cell).
    orbit_panels = (["3d"] if opts["orbit_3d"] else []) + list(planes)
    error_panels = [(key, label, name, colour) for key, label, name, colour in [
        ("H_rel", r"$H_{rel}=|H_n-H_0|/|H_0|$", "Relative Hamiltonian drift", "tab:blue"),
        ("P_rel", r"$P_{rel}=|\hat{P}_n-\hat{P}_0|/P_{ref}$", "Relative linear momentum error", "tab:green"),
        ("L_rel", r"$L_{rel}=|\hat{L}_n-\hat{L}_0|/L_{ref}$", "Relative angular momentum error", "tab:red"),
    ] if opts[key.lower()]]
    rows = [panels for panels in (orbit_panels, error_panels) if panels]
    if not rows:
        return None     # nothing selected: no empty figure is produced

    traj = log_trajectories(log)        # shape: (N, frames, 3)
    shown = choose_bodies_plot(traj, opts)
    t_axis = log_column(log, "t") / t_unit

    # One colour per drawn body. Bodies that share an orbit (e.g. the figure-eight) would hide one
    # another behind lines of equal width, so with up to five bodies the widths decrease from the
    # first body to the last, keeping every trajectory visible as a band inside the previous one.
    colours = plt.cm.tab10(np.arange(len(shown)) % 10) if len(shown) <= 10 else plt.cm.turbo(np.linspace(0, 1, len(shown)))
    widths = np.linspace(3.0, 0.8, len(shown)) if 1 < len(shown) <= 5 else np.full(len(shown), 1.2)

    # Common limits for every orbit view: a cube with a 10% margin around the furthest point of the
    # drawn orbits, so that lengths along different axes are directly comparable
    lim = 1.1 * np.abs(traj[shown]).max()
    final = traj[:, -1]     # final positions of ALL bodies (drawn as grey dots when only a subset is shown)
    draw_rest = len(shown) < log["N"]
    inside = np.all(np.abs(final) < lim, axis=1)        # 3D axes do not clip, so far-away bodies are left out

    # Each row is a sub-figure with its own number of columns, so that e.g. one 3D view and two
    # projections on top can sit above three conservation panels underneath
    fig = plt.figure(figsize=(16, 5.5 * len(rows)), layout="constrained")
    subfigs = fig.subfigures(len(rows), 1, squeeze=False)[:, 0]

    for subfig, panels in zip(subfigs, rows):
        for idx, panel in enumerate(panels):

            # 3D view: the end markers (o) lie on top of the start markers (x) for a periodic orbit
            if panel == "3d":
                ax = subfig.add_subplot(1, len(panels), idx + 1, projection="3d")
                if draw_rest:
                    ax.scatter(*final[inside].T, color="grey", s=6, alpha=0.5, label="all bodies (final)")
                for c, k in enumerate(shown):
                    x, y, z = traj[k].T     # transpose (frames, 3) into three coordinate arrays
                    ax.plot(x, y, z, color=colours[c], lw=widths[c], label=f"Body {k + 1}")
                    ax.scatter(x[0], y[0], z[0], color=colours[c], marker="x", s=30)     # starting position
                    ax.scatter(x[-1], y[-1], z[-1], color=colours[c], s=30)     # final position
                ax.set_xlim(-lim, lim)
                ax.set_ylim(-lim, lim)
                ax.set_zlim(-lim, lim)
                ax.set_box_aspect((1, 1, 1))
                ax.set_xlabel("x")
                ax.set_ylabel("y")
                ax.set_zlabel("z")
                ax.set_title("Orbits in 3D (x = start, o = end)")

            # 2D projection: equal aspect ratio, so that the true shape of the orbit is preserved
            elif panel in plot_planes:
                i, j = plot_planes[panel]
                ax = subfig.add_subplot(1, len(panels), idx + 1)
                if draw_rest:
                    ax.scatter(final[:, i], final[:, j], color="grey", s=6, alpha=0.5, label="all bodies (final)")
                for c, k in enumerate(shown):
                    ax.plot(traj[k, :, i], traj[k, :, j], color=colours[c], lw=widths[c], label=f"Body {k + 1}")
                    ax.scatter(traj[k, 0, i], traj[k, 0, j], color=colours[c], marker="x", s=40, zorder=3)
                    ax.scatter(traj[k, -1, i], traj[k, -1, j], color=colours[c], s=40, zorder=3)
                ax.set_xlim(-lim, lim)
                ax.set_ylim(-lim, lim)
                ax.set_aspect("equal")
                ax.set_xlabel(panel[0])
                ax.set_ylabel(panel[1])
                ax.set_title(f"Orbits projected onto the {panel[0]}-{panel[1]} plane")
                ax.grid(alpha=0.3)

            # Conservation panel. The floor of 1e-18 only keeps exact zeros (e.g. at t = 0) drawable on
            # a logarithmic axis, as in Experiments 1 to 3.
            else:
                key, ylabel, name, colour = panel
                ax = subfig.add_subplot(1, len(panels), idx + 1)
                values = log_column(log, key)
                if opts["log_scale"]:
                    ax.semilogy(t_axis, np.maximum(values, 1e-18), color=colour, lw=1.2)
                    # The exact zero at t = 0 would otherwise stretch the axis down to the floor and squash
                    # the actual error curve, so the axis starts one decade below the smallest non-zero value
                    positive = values[values > 0]
                    if positive.size:
                        ax.set_ylim(bottom=max(1e-18, positive.min() / 10))
                    ax.grid(alpha=0.3, which="both")
                else:
                    ax.plot(t_axis, values, color=colour, lw=1.2)
                    ax.grid(alpha=0.3)

                # Hot-swap schedules: shade every integrator segment and name it at the top of the panel
                if len(log["segments"]) > 1:
                    bounds = [row for row, _ in log["segments"]] + [log["n_rows"] - 1]
                    for s, (row, label) in enumerate(log["segments"]):
                        t_a, t_b = t_axis[bounds[s]], t_axis[bounds[s + 1]]
                        ax.axvspan(t_a, t_b, color=plt.cm.tab10(s % 10), alpha=0.08, lw=0)
                        ax.text(0.5 * (t_a + t_b), 1.0, label, transform=ax.get_xaxis_transform(),
                                ha="center", va="bottom", fontsize=8)

                ax.set_xlabel(f"time ({t_unit_name})")
                ax.set_ylabel(ylabel)
                ax.set_title(name, pad=14 if len(log["segments"]) > 1 else 6)

            # Legends only on the orbit views, where the body colours need explaining. With more than
            # ten entries the legend would cover the plot, so it is left out.
            if not isinstance(panel, tuple) and len(shown) + int(draw_rest) <= 11:
                ax.legend(fontsize=8, loc="upper left")

    fig.suptitle(title)
    return fig

In [ ]:
"""
JSON Experiment Loader (08/11/26).
Experiments 1 to 3 are each written as one long cell in which the parameters, the initial conditions,
the simulation loop and the plots are interleaved. The cells below separate WHAT is simulated from HOW
it is simulated: an experiment is described in a JSON file, and the runtime cell turns that description
into a simulation, a log and a set of plots. The pipeline therefore is

    experiment.json  ->  load_experiment  ->  run_experiment  ->  log (np.array table)  +  plots

This cell contains the loader: it reads the JSON file, checks it, fills in the default values and
translates the names used in the file ("VV4", "barnes_hut", "plummer", ...) into the functions of the
notebook. The structure of the file is documented in README.md and illustrated by the files in the
experiments/ folder. In short, an experiment file holds the following blocks:

    name, description – what the experiment is (free text)
    seed – seed of the random generator (only matters for randomised initial conditions)
    G, softening – constant of gravitational attraction and Plummer softening length (Eq. 4)
    integrator – one integrator key, or a list of keys for a hot-swap schedule (as in Experiment 3)
    force – direct summation or Barnes-Hut with a given opening angle theta
    time – duration (in a chosen time unit), number of timesteps and number of stored frames
    bodies – the initial conditions: an explicit list of bodies, a Plummer sphere or a random sphere
    plots – which plots are included in the output (see plot_defaults in the cell above)
    report – optional extra checks printed after the run (e.g. the closure error of Experiment 1)

JSON has no comment syntax, so every key that starts with an underscore (e.g. "_comment") is ignored
by the loader and can be used to annotate the file.
"""

# The modules json and copy are imported in the first cell, the Body class and the generators are
# defined in the helper cells, and the integrator keys are those of integrator_registry.


# Allowed keys of every block of the file: (required keys, optional keys with their default values).
# Any key that is neither required nor optional is rejected, so that a typo such as "orbit3d" raises an
# error instead of being silently ignored (which would quietly fall back on the default).
schema_experiment = (
    ["name", "integrator", "time", "bodies"],
    {"description": "", "seed": 256, "G": 1.0, "softening": 0.0, "force": {"method": "direct"},
     "plots": {}, "report": {}},
)
schema_time = (
    ["duration", "n_steps", "n_frames"],
    {"unit": 1.0, "unit_name": "simulation time units"},
)
schema_force = {
    "direct": ([], {}),
    "barnes_hut": ([], {"theta": 0.5}),
}
schema_bodies = {
    "explicit": (["list"], {"centre_of_mass_frame": False}),
    "plummer": (["N"], {"M": 1.0, "a": None, "r_cut": 10.0, "virialise": True}),
    "random_sphere": (["N"], {"M": 1.0, "R": 1.2, "mass_spread": 0.5, "q_virial": 0.5}),
}
schema_body = (["m", "r", "v"], {})
schema_plots = ([], dict(plot_defaults))
schema_report = ([], {"closure": False})


# Check one block of the file against its schema and fill in the defaults.
# block – the dictionary read from the file
# schema – (required keys, optional keys with defaults)
# where – name of the block, used in the error messages
# Returns a new dictionary: the original block is left untouched.
def check_block(block, schema, where):
    required, optional = schema
    if not isinstance(block, dict):
        raise TypeError(f"'{where}' must be a JSON object, got {type(block).__name__}")

    keys = {k for k in block if not k.startswith("_")}      # "_comment" style keys are annotations
    missing = [k for k in required if k not in keys]
    unknown = sorted(keys - set(required) - set(optional))
    if missing:
        raise ValueError(f"'{where}' is missing the required key(s): {', '.join(missing)}")
    if unknown:
        raise ValueError(f"'{where}' has unknown key(s): {', '.join(unknown)} "
                       f"(allowed: {', '.join(list(required) + list(optional))})")

    out = copy.deepcopy(optional)       # defaults first ...
    out.update({k: copy.deepcopy(block[k]) for k in keys})      # ... overwritten by the file
    return out


# Check that a value is a positive number (used for every physical parameter that must be > 0)
def check_positive(value, where, integer=False):
    kind = int if integer else (int, float)
    if isinstance(value, bool) or not isinstance(value, kind) or value <= 0:
        raise ValueError(f"'{where}' must be a positive {'integer' if integer else 'number'}, got {value!r}")


# Read an experiment and return its complete, checked description.
# source – path of a JSON file, or a dictionary with the same structure (convenient for quick tests
#          written directly in the notebook)
def load_experiment(source):
    # Read the file (or copy the dictionary, so that the caller's object is never modified)
    if isinstance(source, dict):
        raw, origin = copy.deepcopy(source), "<dict>"
    else:
        origin = str(source)
        with open(source, "r", encoding="utf-8") as f:
            raw = json.load(f)

    cfg = check_block(raw, schema_experiment, "experiment")
    cfg["source"] = origin

    # Integrator: a single key, or a list of keys that take over one after another (hot-swap)
    schedule = cfg["integrator"] if isinstance(cfg["integrator"], list) else [cfg["integrator"]]
    if not schedule:
        raise ValueError("'integrator' must not be an empty list")
    for key in schedule:
        if not isinstance(key, str) or key not in integrator_registry:
            raise ValueError(f"unknown integrator '{key}' (available: {', '.join(integrator_registry)})")
    cfg["schedule"] = schedule

    # Force routine
    if not isinstance(cfg["force"], dict):
        raise TypeError(f"'force' must be a JSON object such as {{\"method\": \"direct\"}}, got {cfg['force']!r}")
    method = cfg["force"].get("method", "direct")
    if method not in schema_force:
        raise ValueError(f"unknown force method '{method}' (available: {', '.join(schema_force)})")
    force = check_block({k: v for k, v in cfg["force"].items() if k != "method"}, schema_force[method], "force")
    if method == "barnes_hut":
        if isinstance(force["theta"], bool) or not isinstance(force["theta"], (int, float)) or force["theta"] < 0:
            raise ValueError(f"'force.theta' must be a non-negative number, got {force['theta']!r}")
    cfg["force"] = {"method": method, **force}

    # Physical constants
    check_positive(cfg["G"], "G")
    if isinstance(cfg["softening"], bool) or not isinstance(cfg["softening"], (int, float)) or cfg["softening"] < 0:
        raise ValueError(f"'softening' must be a non-negative number, got {cfg['softening']!r}")

    # Time discretisation, with the same three controls as in Experiments 1 to 3:
    #   duration * unit – total simulated time t_total
    #   n_steps – number of timesteps, which alone dictates dt = t_total / n_steps
    #   n_frames – number of stored frames (independent of dt)
    tm = check_block(cfg["time"], schema_time, "time")
    check_positive(tm["duration"], "time.duration")
    check_positive(tm["unit"], "time.unit")
    check_positive(tm["n_steps"], "time.n_steps", integer=True)
    check_positive(tm["n_frames"], "time.n_frames", integer=True)

    # Every frame must hold a whole number of steps, and every integrator segment a whole number of
    # frames, so that the run ends exactly at t_total and every integrator covers the same time
    if tm["n_steps"] % tm["n_frames"] != 0:
        raise ValueError(f"time.n_steps ({tm['n_steps']}) must be divisible by time.n_frames ({tm['n_frames']})")
    if tm["n_frames"] % len(schedule) != 0:
        raise ValueError(f"time.n_frames ({tm['n_frames']}) must be divisible by the number of "
                         f"integrator segments ({len(schedule)})")
    tm["t_total"] = tm["duration"] * tm["unit"]
    tm["dt"] = tm["t_total"] / tm["n_steps"]
    tm["steps_per_frame"] = tm["n_steps"] // tm["n_frames"]
    tm["frames_per_segment"] = tm["n_frames"] // len(schedule)
    cfg["time"] = tm

    # Initial conditions
    if not isinstance(cfg["bodies"], dict):
        raise TypeError(f"'bodies' must be a JSON object with a \"type\" key, got {type(cfg['bodies']).__name__}")
    kind = cfg["bodies"].get("type")
    if kind not in schema_bodies:
        raise ValueError(f"unknown bodies type {kind!r} (available: {', '.join(schema_bodies)})")
    bodies = check_block({k: v for k, v in cfg["bodies"].items() if k != "type"}, schema_bodies[kind], "bodies")
    if kind == "explicit":
        if not isinstance(bodies["list"], list) or len(bodies["list"]) < 2:
            raise ValueError("'bodies.list' must hold at least two bodies")
        for k, b in enumerate(bodies["list"]):
            b = check_block(b, schema_body, f"bodies.list[{k}]")
            check_positive(b["m"], f"bodies.list[{k}].m")
            for vec in ("r", "v"):
                if not isinstance(b[vec], list) or len(b[vec]) != 3:
                    raise ValueError(f"'bodies.list[{k}].{vec}' must be a list of 3 numbers (x, y, z)")
            bodies["list"][k] = b
    else:
        check_positive(bodies["N"], "bodies.N", integer=True)
        if bodies["N"] < 2:
            raise ValueError("'bodies.N' must be at least 2")
        check_positive(bodies["M"], "bodies.M")
    cfg["bodies"] = {"type": kind, **bodies}

    # Plots and report: only the keys are checked here; the plotter interprets the values
    cfg["plots"] = check_block(cfg["plots"], schema_plots, "plots")
    if cfg["plots"]["orbit_2d"] is True:
        cfg["plots"]["orbit_2d"] = ["xy"]       # "orbit_2d": true is shorthand for the x-y projection
    for plane in cfg["plots"]["orbit_2d"] or []:
        if plane not in plot_planes:
            raise ValueError(f"unknown projection plane '{plane}' (available: {', '.join(plot_planes)})")
    check_positive(cfg["plots"]["max_bodies"], "plots.max_bodies", integer=True)
    hl = cfg["plots"]["highlight"]
    if hl is not None and (not isinstance(hl, list) or not hl):
        raise ValueError(f"'plots.highlight' must be null or a non-empty list of body numbers, got {hl!r}")
    cfg["report"] = check_block(cfg["report"], schema_report, "report")

    return cfg


# Build the force routine named in the experiment (acc_bh_theta fixes the opening angle).
def build_force(cfg):
    if cfg["force"]["method"] == "direct":
        return acc
    return acc_bh_theta(cfg["force"]["theta"])


# Build the initial conditions named in the experiment.
# The random generator is re-seeded first, so that a randomised experiment produces the same bodies
# every time it is run, regardless of which cells ran before (as in Experiments 2 and 3).
def build_bodies(cfg):
    rnd.seed(cfg["seed"])
    spec, G, e = cfg["bodies"], cfg["G"], cfg["softening"]

    # Explicit list: every body is given by its mass, displacement and velocity, optionally moved
    # to the centre-of-mass frame, so that the system does not drift and P_0 = 0
    if spec["type"] == "explicit":
        bod_obj = [Body(b["m"], b["r"], b["v"]) for b in spec["list"]]
        if spec["centre_of_mass_frame"]:
            to_com_frame(bod_obj)
        return bod_obj

    # Plummer sphere (as in Experiment 2). The default scale length a = 3 pi G M^2 / 16 gives the
    # total energy E = -1/4 (Henon units).
    if spec["type"] == "plummer":
        a = spec["a"] if spec["a"] is not None else 3.0 * np.pi * G * spec["M"]**2 / 16.0
        bod_obj = init_plummer(spec["N"], G, spec["M"], a, spec["r_cut"])
        if spec["virialise"]:
            virialise(bod_obj, G, e)
        return bod_obj

    # Uniform random sphere with unequal masses (as in Experiment 3)
    if spec["type"] == "random_sphere":
        return init_random_sphere(spec["N"], G, spec["M"], spec["R"], e, spec["mass_spread"], spec["q_virial"])

In [ ]:
"""
Experiment Runtime (08/11/26).
The runtime combines the loader, the logger and the plotter into a single call:

    result = run_experiment("experiments/figure_eight.json")

which
    1. loads and checks the experiment file (load_experiment)
    2. builds the initial conditions and the force routine (build_bodies, build_force)
    3. creates the log and records the initial state (init_log)
    4. advances the bodies frame by frame with the scheduled integrator(s), logging every frame (log_state)
    5. prints a summary and draws the plots selected in the file (log_summary, plot_log)
and returns everything that was produced, so that the log can be analysed further in later cells.

The simulation loop is the one of Experiments 1 to 3: the run is divided into n_frames equal blocks of
steps_per_frame steps, and one row is logged at the end of every block. A hot-swap schedule simply
divides the frames into equal segments, one per integrator, which all advance the SAME list of bodies
in place (see the explanation of Experiment 3 for why the swap is exact).
"""


# Run one experiment.
# source – path of a JSON experiment file, or a dictionary with the same structure
# show – draw the selected plots (False is useful for batch runs or timing)
# Returns a dictionary with:
#   config – the complete, checked experiment description (with all defaults filled in)
#   log – the log of the run (see init_log); log["table"] is the np.array table of the results
#   bodies – the bodies in their final state
#   initial – an independent copy of the initial conditions
#   runtime – wall-clock time spent inside the integrators (s), as in Experiments 2 and 3
#   figure – the matplotlib figure, or None if no plot was selected or show = False
def run_experiment(source, show=True):
    cfg = load_experiment(source)
    tm, G, e = cfg["time"], cfg["G"], cfg["softening"]

    # Initial conditions and force routine
    bod_obj = build_bodies(cfg)
    initial = copy_bodies(bod_obj)      # kept for the closure test and for re-runs
    force = build_force(cfg)

    # The explicit body numbers of the plot selection can only be checked once N is known
    highlight = cfg["plots"]["highlight"]
    if highlight is not None and not all(isinstance(k, int) and 1 <= k <= len(bod_obj) for k in highlight):
        raise ValueError(f"'plots.highlight' must list body numbers between 1 and {len(bod_obj)}, got {highlight}")

    # Header: everything needed to reproduce the run, in the style of the experiment printouts
    force_name = "direct summation" if cfg["force"]["method"] == "direct" else f"Barnes-Hut, theta = {cfg['force']['theta']}"
    print(f"=== {cfg['name']} ({cfg['source']}) ===")
    if cfg["description"]:
        print(cfg["description"])
    print(f"N = {len(bod_obj)}, G = {G}, softening = {e}, force: {force_name}")
    print(f"integrator: {' -> '.join(cfg['schedule'])}, t_total = {tm['t_total']:.4f} "
          f"({tm['duration']} {tm['unit_name']}), n_steps = {tm['n_steps']}, dt = {tm['dt']:.3e}")
    print(f"n_frames = {tm['n_frames']}, steps_per_frame = {tm['steps_per_frame']}")

    # Log with the initial state as row 0
    log = init_log(G, e, bod_obj, tm["n_frames"])

    # Simulation
    # The simulated time of every frame is computed from the frame counter (frame * steps_per_frame * dt)
    # rather than accumulated, so that no rounding error builds up in the time column.
    runtime = 0.0
    frame = 0
    for key in cfg["schedule"]:
        run = integrator_registry[key]["run"]       # the integrator that takes over at this boundary
        log_segment(log, key)

        for _ in range(tm["frames_per_segment"]):
            # Only the time spent inside the integrator is measured as the runtime
            clock = time.perf_counter()
            run(bod_obj, G, e, tm["dt"], tm["steps_per_frame"], force)
            runtime += time.perf_counter() - clock

            frame += 1
            log_state(log, bod_obj, frame * tm["steps_per_frame"] * tm["dt"])

    # Summary
    print(f"runtime = {runtime:.2f} s")
    log_summary(log)

    # Closure check (Experiment 1): after a whole number of periods every body should be back where it
    # started, so |r(t_total) - r(0)| measures the accumulated error. Only meaningful for periodic orbits.
    if cfg["report"]["closure"]:
        for k, (obj, obj_0) in enumerate(zip(bod_obj, initial)):
            print(f"body {k + 1}: closure error |r(T) - r(0)| = {np.linalg.norm(obj.r - obj_0.r):.3e}")

    # Plots selected in the file
    fig = None
    if show:
        # A single integrator is named in full; a hot-swap schedule by its keys, to keep the title on one line
        if len(cfg["schedule"]) == 1:
            scheme = integrator_registry[cfg["schedule"][0]]["label"]
        else:
            scheme = "hot-swap " + " -> ".join(cfg["schedule"])
        title = (f"{cfg['name']}: {tm['duration']} {tm['unit_name']}, {tm['n_steps']} steps, "
                 f"dt = {tm['dt']:.2e} ({scheme}, {force_name})")
        fig = plot_log(log, cfg["plots"], title=title, t_unit=tm["unit"], t_unit_name=tm["unit_name"])
        if fig is not None:
            plt.show()

    return {"config": cfg, "log": log, "bodies": bod_obj, "initial": initial, "runtime": runtime, "figure": fig}


# Run several experiments one after another and collect their results by name.
# sources – list of JSON paths (or dictionaries), e.g. sorted(Path("experiments").glob("*.json"))
def run_experiments(sources, show=True):
    results = {}
    for source in sources:
        res = run_experiment(source, show=show)
        name = res["config"]["name"]
        if name in results:     # two files with the same name: keep both, told apart by their path
            name = f"{name} ({res['config']['source']})"
        results[name] = res
        print()
    return results

In [ ]:
"""
Running JSON Experiments (08/11/26).
Example use of the runtime. Experiment 1 has been rewritten as experiments/figure_eight.json, and running
it through the runtime reproduces the printout of the Experiment 1 cell exactly (same closure errors and
same maximum relative errors), which validates the logger against the hand-written loop.
The other files of the experiments/ folder are described in README.md.
"""

# Run one experiment. The result is assigned to a variable: a bare run_experiment(...) as the last line
# of a cell would make Jupyter print the whole result dictionary, including the log table.
result_fe = run_experiment("experiments/figure_eight.json")

# Working with the log table
# Every row is one stored frame, every column one logged quantity (see log["columns"]). Single columns
# are read by name, and the trajectories are returned in the (body, frame, coordinate) layout of the
# experiment cells.
log_fe = result_fe["log"]
print(f"\nlog table: {log_fe['table'].shape[0]} rows (frames) x {log_fe['table'].shape[1]} columns")
print("columns:", ", ".join(log_fe["columns"]))
print(f"final H_rel = {log_column(log_fe, 'H_rel')[-1]:.3e}, trajectories: {log_trajectories(log_fe).shape}")

# Run every experiment of the folder in turn (uncomment). Together they take about a minute.
# results = run_experiments(sorted(Path("experiments").glob("*.json")))

## References
- Aarseth, S. J., Hénon, M. and Wielen, R. (1974) A comparison of numerical methods for the study of star cluster dynamics. *Astronomy and Astrophysics*, 37, 183–187.
- Barnes, J. and Hut, P. (1986) A hierarchical $\mathcal{O}(N\log N)$ force-calculation algorithm. *Nature*, 324, 446–449.
- Chenciner, A. and Montgomery, R. (2000) A remarkable periodic solution of the three-body problem in the case of equal masses. *Annals of Mathematics*, 152(3), 881–901.
- Dehnen, W. and Read, J. I. (2011) N-body simulations of gravitational dynamics. *The European Physical Journal Plus*, 126, 55.
- Forest, E. and Ruth, R. D. (1990) Fourth-order symplectic integration. *Physica D: Nonlinear Phenomena*, 43(1), 105–117.
- Hairer, E., Lubich, C. and Wanner, G. (2006) *Geometric Numerical Integration: Structure-Preserving Algorithms for Ordinary Differential Equations*. 2nd edn. Berlin: Springer.
- Hairer, E., Nørsett, S. P. and Wanner, G. (1993) *Solving Ordinary Differential Equations I: Nonstiff Problems*. 2nd edn. Berlin: Springer.
- Kutta, W. (1901) Beitrag zur näherungsweisen Integration totaler Differentialgleichungen. *Zeitschrift für Mathematik und Physik*, 46, 435–453.
- Moore, C. (1993) Braids in classical dynamics. *Physical Review Letters*, 70(24), 3675–3679.
- Omelyan, I. P., Mryglod, I. M. and Folk, R. (2002) Optimized Forest–Ruth- and Suzuki-like algorithms for integration of motion in many-body systems. *Computer Physics Communications*, 146(2), 188–202.
- Plummer, H. C. (1911) On the problem of distribution in globular star clusters. *Monthly Notices of the Royal Astronomical Society*, 71, 460–470.
- Runge, C. (1895) Über die numerische Auflösung von Differentialgleichungen. *Mathematische Annalen*, 46, 167–178.
- Salmon, J. K. and Warren, M. S. (1994) Skeletons from the treecode closet. *Journal of Computational Physics*, 111(1), 136–155.
- Simó, C. (2002) Dynamical properties of the figure eight solution of the three-body problem. In: Chenciner, A., Cushman, R., Robinson, C. and Xia, Z. J. (eds) *Celestial Mechanics: Dedicated to Donald Saari for his 60th Birthday*. Contemporary Mathematics, 292. Providence: American Mathematical Society, 209–228.
- Springel, V. (2005) The cosmological simulation code GADGET-2. *Monthly Notices of the Royal Astronomical Society*, 364(4), 1105–1134.
- Swope, W. C., Andersen, H. C., Berens, P. H. and Wilcox, K. R. (1982) A computer simulation method for the calculation of equilibrium constants for the formation of physical clusters of molecules: application to small water clusters. *The Journal of Chemical Physics*, 76(1), 637–649.
- Verlet, L. (1967) Computer "experiments" on classical fluids. I. Thermodynamical properties of Lennard-Jones molecules. *Physical Review*, 159(1), 98–103.
- Yoshida, H. (1990) Construction of higher order symplectic integrators. *Physics Letters A*, 150(5–7), 262–268.